FILMMAKERS NODE GRAPH!

Importar paquetes

In [1]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import math

Abrir dataframe original

In [2]:
SHEET_ID = "19cpbDSusF5LmknZB-mmiWKk-oibUxwzYX-yZdw_rl4Q"
GIDS = {
    "MOVIES": 734211922,
    "DIRECTORS": 756156538,
    "RELATIONS": 195959194,
}


def _read_sheet_csv(gid):
    url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={gid}"
    return pd.read_csv(url)


# DIRECTORS y RELATIONS ya vienen calculados por formulas nativas de Sheets
# (SUMIF/COUNTIF/XLOOKUP, las mismas que ten\u00eda directors_favs_rev.xlsx),
# no en pandas. La base de datos \u00fanica del proyecto es la hoja "elo" de
# Google Sheets.
dirdf = _read_sheet_csv(GIDS["DIRECTORS"])
moviedf = _read_sheet_csv(GIDS["MOVIES"])
relatedf = _read_sheet_csv(GIDS["RELATIONS"])

dirdf["director"] = dirdf["director"].astype(str)
moviedf["movie"] = moviedf["movie"].astype(str)

# El CSV exportado desde Sheets trae rating/diary_count/year como texto en
# cuanto hay alguna fila en blanco (peliculas sin ver todavia); el resto del
# notebook los necesita numericos.
moviedf["rating"] = pd.to_numeric(moviedf["rating"], errors="coerce").fillna(0)
moviedf["diary_count"] = pd.to_numeric(moviedf["diary_count"], errors="coerce").fillna(0)
moviedf["year"] = pd.to_numeric(moviedf["year"], errors="coerce")


Creación del dataframe de influencias

In [3]:
# Normalizar nombres de RELATIONS contra los de MOVIES/DIRECTORS antes de construir
# el grafo: pequeñas diferencias de mayúsculas ("Goodfellas" vs "GoodFellas",
# "Call Me By Your Name" vs "Call Me by Your Name", etc.) hacían que se creara un
# nodo "fantasma" extra (sin imagen ni tipo) en vez de conectarse al nodo real.
_movie_lookup = {str(t).strip().lower(): t for t in moviedf["movie"].dropna()}
_director_lookup = {str(t).strip().lower(): t for t in dirdf["director"].dropna()}

def _normalize_movie(name):
    if pd.isna(name):
        return name
    return _movie_lookup.get(str(name).strip().lower(), name)

def _normalize_director(name):
    if pd.isna(name):
        return name
    return _director_lookup.get(str(name).strip().lower(), name)

relatedf["source_director"] = relatedf["source_director"].apply(_normalize_director)
relatedf["target_director"] = relatedf["target_director"].apply(_normalize_director)
relatedf["source_movie"] = relatedf["source_movie"].apply(_normalize_movie)
relatedf["target_movie"] = relatedf["target_movie"].apply(_normalize_movie)

edges = []
for _, r in relatedf.iterrows():

    sd = r["source_director"]
    sm = r["source_movie"]
    td = r["target_director"]
    tm = r["target_movie"]
    t  = r["type"] if pd.notna(r["type"]) else None

    # --- CASO A: NO hay películas → director → director
    if pd.isna(sm) and pd.isna(tm):
        edges.append({
            "from": sd,
            "to": td,
            "label": "dtd",
            "type": t,
        })

    # --- CASO B: hay películas → director → movie → movie → director
    else:
        if pd.notna(sm):
            edges.append({
                "from": sd,
                "to": sm,
                "label": "dtm",
                "type": t,
            })

        if pd.notna(sm) and pd.notna(tm):
            edges.append({
                "from": sm,
                "to": tm,
                "label": "mtm",
                "type": t,
            })

        if pd.notna(tm):
            edges.append({
                "from": tm,
                "to": td,
                "label": "dtm",
                "type": t,
            })
edges_df = pd.DataFrame(edges)

# Un "dtm" (director<->película, en cualquiera de los dos sentidos: el
# director hizo la película, o la película fue hecha por ese director) se
# genera una vez por cada fila de RELATIONS que menciona esa película — una
# película que participa en muchas relaciones terminaba con el mismo par
# director-película repetido tantas veces como filas la mencionan (ej.
# Chaplin -> City Lights aparecía 4 veces). Es un solo hecho, no una
# relación por fila, así que se deduplica por (from, to) quedándose con una
# fila que tenga type por sobre una en blanco.
_dtm_mask = edges_df["label"] == "dtm"
_dtm = edges_df[_dtm_mask].sort_values("type", na_position="last")
_dtm = _dtm.drop_duplicates(subset=["from", "to"], keep="first")
edges_df = pd.concat([edges_df[~_dtm_mask], _dtm], ignore_index=True)

Rango de tamaños de nodos, de colores y años

In [4]:
#tamaños minimo y maximo de los nodos (duplicados)
min_size = 8
max_size = 120

dir_shown_mask = dirdf["references"] > 0
dirdf["node_size"] = float(min_size)
dirdf.loc[dir_shown_mask, "node_size"] = (
    dirdf.loc[dir_shown_mask, "rating_sum"].rank(pct=True, method="average")
) * (max_size - min_size) + min_size
dirdf["node_size"] = dirdf["node_size"].fillna(min_size)

moviedf["score"] = pd.to_numeric(moviedf["rating"], errors="coerce") * moviedf["diary_count"]
shown_mask = moviedf["references"] > 0
moviedf["node_size"] = float(min_size)
# gamma < 1 achica menos las películas de rating/score bajo (2.5-3 quedaban
# casi en min_size): sqrt del percentil las agranda relativo a la escala
# lineal, sin tocar el tope (rating 5 sigue llegando a max_size).
size_gamma = 0.5
moviedf.loc[shown_mask, "node_size"] = (
    moviedf.loc[shown_mask, "score"].rank(pct=True, method="average") ** size_gamma
) * (max_size - min_size) + min_size
moviedf["node_size"] = moviedf["node_size"].fillna(min_size)
# Las peliculas a la mitad del tamano de un director equivalente: sin esto
# competian visualmente con los directores en vez de leerse como satelites
# mas chicos alrededor de ellos.
MOVIE_SIZE_SCALE = 0.5
moviedf["node_size"] = moviedf["node_size"] * MOVIE_SIZE_SCALE

# Las peliculas con rating 0 (no vistas) no tienen diary_count, asi que su
# score siempre da 0 y quedan todas empatadas en el percentil mas bajo. En
# vez de leerse como "las peor puntuadas", se les da el tamano promedio de
# las peliculas con el rating real mas bajo que aparece en el grafo (no
# siempre hay alguna con exactamente 2, asi que se toma el minimo real).
_rated_mask = shown_mask & (moviedf["rating"] > 0)
if _rated_mask.any():
    _lowest_rating = moviedf.loc[_rated_mask, "rating"].min()
    UNWATCHED_TARGET_SIZE = moviedf.loc[
        _rated_mask & (moviedf["rating"] == _lowest_rating), "node_size"
    ].mean()
    moviedf.loc[shown_mask & (moviedf["rating"] == 0), "node_size"] = UNWATCHED_TARGET_SIZE

COUNTRY_COLORS = {
    "USA":          "#38bdf8",
    "UK":           "#f43f5e",
    "France":       "#c084fc",
    "Japan":        "#fbbf24",
    "Chile":        "#4ade80",
    "Italy":        "#fb923c",
    "Germany":      "#94a3b8",
    "Russia":       "#818cf8",
    "Czechia":      "#22d3ee",
    "Poland":       "#f472b6",
    "Canada":       "#e879f9",
    "Sweden":       "#fde047",
    "Spain":        "#f97316",
    "China":        "#e11d48",
    "Argentina":    "#67e8f9",
    "South Korea":  "#ec4899",
    "Taiwan":       "#fb7185",
    "India":        "#d97706",
    "Iran":         "#65a30d",
    "Denmark":      "#fca5a5",
    "Norway":       "#bfdbfe",
    "Finland":      "#a5f3fc",
    "Netherlands":  "#7dd3fc",
    "Belgium":      "#fef08a",
    "Austria":      "#f9a8d4",
    "Greece":       "#fdba74",
    "Ireland":      "#86efac",
    "Serbia":       "#6366f1",
    "Brazil":       "#a3e635",
    "Mexico":       "#2dd4bf",
    "Uruguay":      "#c7d2fe",
    "Cuba":         "#fda4af",
    "South Africa": "#84cc16",
    "Senegal":      "#fdcfe8",
    "Israel":       "#60a5fa",
    "New Zealand":  "#a1a1aa",
    "Australia":    "#a78bfa",
    "Latvia":       "#d1fae5",
}

DEFAULT_COLOR = "#9ca3af"

# ---- Estilo de líneas de influencia por tipo ----
# RELATIONS.type toma uno de 8 valores (ver hoja "extra"). Cada tipo tiene un
# color y un patrón de guiones (dashes) propios para poder distinguirse a
# simple vista, incluso en escala de grises, sin depender solo del color.
# Las relaciones que todavía no tienen tipo asignado (la mayoría hoy) usan un
# gris plano y translúcido para no competir visualmente con las clasificadas.
def hex_to_rgba(hex_color, alpha):
    hex_color = hex_color.lstrip("#")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

# Mismo grosor para el marco de los nodos (net.set_options -> nodes.borderWidth,
# 3px) y para TODAS las lineas de conexion: asi se sienten parte del mismo
# lenguaje visual en vez de grosores inconsistentes entre si. Los dashes ya
# quedan comodamente mas grandes que este ancho (ver nota abajo), asi que
# unificar el width no le quita legibilidad al patron de cada tipo.
FRAME_AND_LINE_WIDTH = 15

TYPE_STYLE_DEFS = {
    # Los segmentos de "dashes" están en px y tienen que ser notoriamente más
    # grandes que el propio "width" de la arista: con un dash de 1-3px sobre
    # una línea de 5-9px de grosor, vis-network dibuja algo indistinguible de
    # una línea sólida (el patrón queda más chico que el grosor del trazo).
    "1-Narrative influence":  {"name": "Influencia narrativa", "hex": "#3987e5", "dashes": False,          "width": FRAME_AND_LINE_WIDTH},
    "2-Technical influence":  {"name": "Influencia técnica",   "hex": "#199e70", "dashes": [26, 12],       "width": FRAME_AND_LINE_WIDTH},
    "3-Direct adaptation":    {"name": "Adaptación directa",   "hex": "#c98500", "dashes": [40, 16],       "width": FRAME_AND_LINE_WIDTH},
    "3-General influence":    {"name": "Influencia general",   "hex": "#06b6d4", "dashes": [8, 8],         "width": FRAME_AND_LINE_WIDTH},
    "4-Specific reference":   {"name": "Referencia puntual",   "hex": "#008300", "dashes": False,          "width": FRAME_AND_LINE_WIDTH},
    "5-Formative director":   {"name": "Director formativo",   "hex": "#9085e9", "dashes": [6, 14],        "width": FRAME_AND_LINE_WIDTH},
    "6-Favorite movie":       {"name": "Película favorita",    "hex": "#e66767", "dashes": [12, 10],       "width": FRAME_AND_LINE_WIDTH},
    "7-Conscious rejection":  {"name": "Rechazo consciente",   "hex": "#d55181", "dashes": [16, 8, 4, 8],  "width": FRAME_AND_LINE_WIDTH},
    "8-Critical analysis":    {"name": "Lectura crítica",      "hex": "#d95926", "dashes": [5, 14],        "width": FRAME_AND_LINE_WIDTH},
}
DEFAULT_EDGE_STYLE_DEF = {"name": "Sin tipo aún", "hex": "#94a3b8", "dashes": False, "width": FRAME_AND_LINE_WIDTH}

# rgba con alfa ya resuelto — lo que consume directamente pyvis/vis.js
TYPE_STYLES = {
    key: {"color": hex_to_rgba(d["hex"], 0.85), "dashes": d["dashes"], "width": d["width"]}
    for key, d in TYPE_STYLE_DEFS.items()
}
DEFAULT_EDGE_STYLE = {
    "color": hex_to_rgba(DEFAULT_EDGE_STYLE_DEF["hex"], 0.35),
    "dashes": False,
    "width": DEFAULT_EDGE_STYLE_DEF["width"],
}

Código para bajar posters

In [5]:
import re as _re
import requests
from pathlib import Path

POSTER_DIR = Path("posters")
POSTER_DIR.mkdir(exist_ok=True)

def sanitize_filename(name: str) -> str:
    """Reemplaza caracteres inválidos en Windows para usar como nombre de archivo."""
    return _re.sub(r'[\\/:*?"<>|]', '_', str(name)).strip()

def get_tmdb_image_local(image_path, local_name, local_dir, size_prefix):
    """
    Descarga y cachea localmente una imagen de TMDB (poster o retrato).
    local_name es el nombre base del archivo (sin extensión), puede ser el
    nombre del director/película o el id. Devuelve la ruta local o None.
    """
    if pd.isna(image_path) or not image_path:
        return None

    if not image_path.startswith("/"):
        image_path = "/" + image_path

    local_path = local_dir / f"{sanitize_filename(local_name)}.jpg"
    if local_path.exists():
        return local_path.as_posix()

    url = f"https://image.tmdb.org/t/p/{size_prefix}{image_path}"

    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            local_path.write_bytes(r.content)
            return local_path.as_posix()
    except requests.RequestException:
        pass

    return None


def get_poster_local(poster_path, movie_name):
    return get_tmdb_image_local(poster_path, movie_name, POSTER_DIR, "w185")


Bajar retratos


In [6]:
PROFILE_DIR = Path("profiles")
PROFILE_DIR.mkdir(exist_ok=True)

def get_director_photo_local(profile_path, person_id):
    # w185 (mismo tamano que los posters) en vez de w300_and_h450_face: ese
    # size especial de TMDB viene pre-recortado bien pegado a la cara, lo que
    # sumado al circulo de vis.js perdia demasiado contexto de la foto. w185
    # es el retrato completo sin ese recorte extra, y de paso pesa menos.
    return get_tmdb_image_local(profile_path, person_id, PROFILE_DIR, "w185")


In [7]:
import json

# Directores fallecidos detectados vía TMDB (/person/{id}.deathday).
# El archivo se genera con fetch_deceased.py; no toca el xlsx.
_deceased_path = Path("deceased_directors.json")
if _deceased_path.exists():
    with open(_deceased_path, encoding="utf-8") as _f:
        deceased_directors = {k for k, v in json.load(_f).items() if v}
else:
    deceased_directors = set()

print(f"Directores fallecidos cargados: {len(deceased_directors)}")


Directores fallecidos cargados: 160


Creación del grafo

In [8]:
# Grafo dirigido (influencias tienen dirección)
G = nx.DiGraph()

# Agregar nodos desde el dataframe original
for _, row in dirdf.iterrows():
    if row["references"] > 0:
        # clave de cache = nombre del director (más legible que el id)
        cache_key = sanitize_filename(row["director"])
        image_path = (
            get_director_photo_local(row["portrait_path"], cache_key)
            if pd.notna(row["portrait_path"]) and row["portrait_path"]
            else None
        )
        G.add_node(
            row["director"],
            country=row["country"],
            year=row["year"],
            size=row["node_size"],
            rating_sum=row["rating_sum"],
            node_type="director",
            image=image_path
        )

for _, row in moviedf.iterrows():
    if row["references"] > 0:
        # clave de cache = título de la película
        cache_key = sanitize_filename(row["movie"])
        image_path = get_poster_local(row["poster_path"], cache_key)
        G.add_node(
            row["movie"],
            node_type="movie",
            references=row["references"],
            year=row.get("year"),
            size=row["node_size"],
            image=image_path,
            genres=str(row["genre"]) if pd.notna(row.get("genre")) else "",
      )

# Agregar aristas desde el dataframe de relaciones
for _, r in edges_df.iterrows():
    G.add_edge(
        r["from"],
        r["to"],
        label=r["label"],
        influence_type=r["type"],
    )

nodes_sorted = sorted(
    G.nodes(data=True),
    key=lambda x: x[1].get("year", 0)
)

Normalizar años float

In [9]:
def normalize_year(y):
    if y is None:
        return None
    if isinstance(y, float):
        if math.isnan(y):
            return None
        return int(round(y))
    try:
        return int(y)
    except (ValueError, TypeError):
        return None

PYVIS Visualización del gráfico

In [10]:

#CREAR PYVIS
net = Network(
    height="900px",
    width="100%",
    bgcolor="#222222",
    font_color="#e5e7eb",
    directed=True
)

# Pais del director PROPIO de cada pelicula (no cualquier director conectado
# por una relacion de influencia) - moviedf ya trae esa columna directamente,
# asi que no hace falta pasar por las relaciones. Se usa para pintar el marco
# de cada poster del mismo color que el pais del director que la hizo.
_director_country = dict(zip(dirdf["director"], dirdf["country"].fillna("Unknown")))
_movie_director = dict(zip(moviedf["movie"], moviedf["director"]))

for node, attrs in nodes_sorted:
    attrs["year"] = normalize_year(attrs.get("year"))
    year = attrs["year"]
    node_type = attrs.get("node_type")
    if node_type == "movie":
        country = _director_country.get(_movie_director.get(node), "Unknown")
    else:
        country = attrs.get("country", "Unknown")
    color = COUNTRY_COLORS.get(country, DEFAULT_COLOR)
    image_path = attrs.get("image")
    if image_path:
        # circularImage recorta el retrato en circulo y dibuja el borde
        # (color.border, mas abajo) como anillo alrededor - para peliculas
        # queda rectangular ("image") para no perder el arte del poster.
        # Cargan de una desde el arranque (no diferido): mejor una carga
        # inicial mas larga que un salto de lentitud despues, cuando se
        # muestran/filtran/clusterizan las peliculas.
        shape = "circularImage" if node_type == "director" else "image"
        image = image_path
    elif node_type == "movie":
        shape = "square"
        image = None
    else:
        shape = "dot"
        image = None
    size = attrs.get("size", 50)
    if image_path:
        # Piso mas chico para peliculas: siguen la misma proporcion que su
        # tamano ya reducido (MOVIE_SIZE_SCALE), en vez de un piso absoluto
        # que las volviera a igualar a un director chico.
        size = max(15 if node_type == "director" else 8, size)
    title = f"{node} ({year}) — {country}" if node_type == "director" else f"{node} ({year})"
    node_kwargs = dict(
        label=node, size=size, base_size=size, node_type=node_type, shape=shape,
        mass=4 if node_type == "director" else 1, year=year,
        country=country if node_type == "director" else None,
        genres=attrs.get("genres", "") if node_type == "movie" else None,
        # El borde usa el mismo color que el fondo: con circularImage se ve
        # como un anillo del color del pais alrededor del retrato/poster.
        color={"background": color, "border": color}, title=title,
    )
    if image is not None:
        node_kwargs["image"] = image
    if node_type == "movie":
        # Ya en la carga inicial: sin esto, la PRIMERA estabilizacion (la que
        # corre pyvis antes de que el JS inyectado alcance a desactivarles la
        # fisica) tiene que mover ~850 peliculas ademas de los directores -
        # es el grueso del costo de "todos los directores" siendo lento.
        node_kwargs["physics"] = False
    net.add_node(node, **node_kwargs)

# Cuando hay más de una arista entre el mismo par de nodos (dos tipos de
# relación entre las mismas películas/directores, o direcciones opuestas)
# vis-network las dibuja superpuestas y no se distinguen. Se cuentan pares
# duplicados de antemano y a cada arista extra se le da curvatura alternada
# (lado y radio distintos) para que se separen visualmente; los pares únicos
# (la gran mayoría) quedan rectos como antes.
_pair_total = {}
for _, r in edges_df.iterrows():
    key = tuple(sorted((r["from"], r["to"]), key=str))
    _pair_total[key] = _pair_total.get(key, 0) + 1
_pair_seen = {}

def edge_smooth(u, v):
    key = tuple(sorted((u, v), key=str))
    if _pair_total.get(key, 0) <= 1:
        return False
    idx = _pair_seen.get(key, 0)
    _pair_seen[key] = idx + 1
    side = "curvedCW" if idx % 2 == 0 else "curvedCCW"
    roundness = 0.15 + 0.12 * (idx // 2)
    return {"enabled": True, "type": side, "roundness": roundness}

for _, r in edges_df.iterrows():
    u, v = r["from"], r["to"]
    label = r["label"]
    itype = r["type"] if pd.notna(r["type"]) else None
    style = TYPE_STYLES.get(itype, DEFAULT_EDGE_STYLE)
    base_color = style["color"]
    base_dashes = style["dashes"]
    width = style["width"]
    smooth = edge_smooth(u, v)
    if label == "dtm":
        # length corto (mas que mtm): una pelicula tiene que quedar pegada a su
        # propio director, no solo "cerca" - es la arista mas importante para
        # que el racimo de filmografia se vea junto.
        net.add_edge(u, v, width=width, dashes=base_dashes, arrows="to", length=220,
                     color=base_color, base_color=base_color, base_dashes=base_dashes,
                     base_width=width, base_physics=True, influence_type=itype, smooth=smooth)
    elif label == "mtm":
        # resorte corto: dos películas que se influyen directamente
        # deberían quedar visualmente cerca una de la otra, no solo
        # unidas por una línea decorativa sin fuerza real. Ni tanto
        # (200) ni tan poco (nada): un punto medio entre "pegadas" y
        # "perdidas en el grafo".
        net.add_edge(u, v, width=width, dashes=base_dashes, length=400, arrows="to",
                     color=base_color, base_color=base_color, base_dashes=base_dashes,
                     base_width=width, base_physics=True, influence_type=itype, smooth=smooth)
    else:
        net.add_edge(u, v, width=width, dashes=base_dashes, length=5400, arrows="to",
                     color=base_color, base_color=base_color, base_dashes=base_dashes,
                     base_width=width, base_physics=True, influence_type=itype, smooth=smooth)

# ----- Hubs de país -----
INVISIBLE = "rgba(0,0,0,0)"
countries_shown = set()
for _, row in dirdf.iterrows():
    if row["references"] > 0:
        countries_shown.add(row.get("country", "Unknown"))

for country in countries_shown:
    net.add_node(
        f"__country_{country}__",
        label=" ", font={"size": 0, "color": INVISIBLE},
        shape="dot", size=1, mass=4, node_type="country_hub",
        color={"background": INVISIBLE, "border": INVISIBLE},
        physics=False,
    )

for _, row in dirdf.iterrows():
    if row["references"] > 0:
        country = row.get("country", "Unknown")
        # length corto a propósito: con 300 el tirón hacia el hub quedaba
        # demasiado débil frente a la repulsión general del grafo y el
        # clustering por país no se notaba.
        net.add_edge(
            row["director"], f"__country_{country}__",
            length=120, width=0, arrows=None,
            color=INVISIBLE, base_color=INVISIBLE, base_physics=True,
        )

# ----- Hubs de género -----
# Una película puede tener varios géneros ("Drama, Comedy"): en vez de
# forzarla a elegir uno "principal", se conecta a CADA uno de sus hubs de
# género. Con un solo género queda pegada a ese hub; con varios, la física
# la asienta en un punto intermedio entre todos - un puente natural entre
# clusters, en vez de un recorte arbitrario.
genres_shown = set()
for _, row in moviedf.iterrows():
    if row["references"] > 0 and pd.notna(row.get("genre")):
        for g in str(row["genre"]).split(","):
            g = g.strip()
            if g:
                genres_shown.add(g)

for genre in genres_shown:
    net.add_node(
        f"__genre_{genre}__",
        label=" ", font={"size": 0, "color": INVISIBLE},
        shape="dot", size=1, mass=4, node_type="genre_hub",
        color={"background": INVISIBLE, "border": INVISIBLE},
        physics=False,
    )

for _, row in moviedf.iterrows():
    if row["references"] > 0 and pd.notna(row.get("genre")):
        for g in str(row["genre"]).split(","):
            g = g.strip()
            if not g:
                continue
            net.add_edge(
                row["movie"], f"__genre_{g}__",
                length=120, width=0, arrows=None,
                color=INVISIBLE, base_color=INVISIBLE, base_physics=True,
            )



Net opciones

In [11]:

net.set_options("""
{
  "layout": {
    "improvedLayout": false
  },
  "nodes": {
    "font": {
      "size": 14,
      "color": "#e5e7eb",
      "strokeWidth": 3,
      "strokeColor": "rgba(0,0,0,0.75)"
    },
    "borderWidth": 3,
    "borderWidthSelected": 5,
    "shadow": {
      "enabled": false,
      "size": 10,
      "x": 2,
      "y": 3
    },
    "shapeProperties": {
      "useBorderWithImage": true
    }
  },
  "edges": {
    "smooth": false
  },
  "interaction": {
    "hover": true,
    "hoverConnectedEdges": true,
    "hideEdgesOnDrag": true,
    "hideEdgesOnZoom": true
  },
  "physics": {
    "enabled": true,
    "solver": "forceAtlas2Based",
    "forceAtlas2Based": {
      "gravitationalConstant": -200,
      "centralGravity": 0.01,
      "springLength": 200,
      "springConstant": 0.18,
      "damping": 0.4,
      "avoidOverlap": 0.8
    },
    "stabilization": {
      "iterations": 500,
      "fit": true
    }
  }
}
""")


In [12]:
import re

dir_years = [normalize_year(attrs.get("year")) for _, attrs in G.nodes(data=True)
             if attrs.get("node_type") == "director" and normalize_year(attrs.get("year"))]
YEAR_MIN = min(dir_years) if dir_years else 1900
YEAR_MAX = max(dir_years) if dir_years else 2025
# El slider usa step=5: si los bordes no son múltiplos de 5, el navegador no
# deja arrastrar hasta el max real (ej. min=1896 con step=5 solo llega a
# 2021, aunque max diga 2025) — se redondean hacia afuera para que el
# rango completo sea siempre alcanzable.
YEAR_STEP = 5
YEAR_MIN = (YEAR_MIN // YEAR_STEP) * YEAR_STEP
YEAR_MAX = -(-YEAR_MAX // YEAR_STEP) * YEAR_STEP

country_colors_json = json.dumps(COUNTRY_COLORS, ensure_ascii=False)
type_styles_json = json.dumps(TYPE_STYLES, ensure_ascii=False)
default_edge_style_json = json.dumps(DEFAULT_EDGE_STYLE, ensure_ascii=False)

cluster_groups = {}
for node, attrs in G.nodes(data=True):
    if attrs.get("node_type") != "director":
        continue
    country = attrs.get("country", "Unknown")
    color = COUNTRY_COLORS.get(country, DEFAULT_COLOR)
    members = {node}
    for nb in G.successors(node):
        if G.get_edge_data(node, nb, {}).get("label") == "dtm":
            members.add(nb)
    for nb in G.predecessors(node):
        if G.get_edge_data(nb, node, {}).get("label") == "dtm":
            members.add(nb)
    cluster_groups[node] = {"color": color, "nodes": sorted(members)}

cluster_groups_json = json.dumps(cluster_groups, ensure_ascii=False)

html = net.generate_html()

header_html = """
<style>
  #graphBgGradient { position:fixed; inset:0; z-index:-1; pointer-events:none;
    background: radial-gradient(ellipse at 50% 35%, #2b2b2b 0%, #1c1c1c 55%, #121212 100%); }
  #graphHeader { display:flex; align-items:center; gap:14px; padding:10px 14px; flex-wrap:wrap; }
  #searchWrap { position:relative; width:250px; }
  #graphHeader input#nodeSearch { width:100%; padding:7px 12px; border-radius:8px;
    border:1px solid #4b5563; background:rgba(20,20,20,0.55); color:#e5e7eb;
    backdrop-filter:blur(6px); -webkit-backdrop-filter:blur(6px); box-sizing:border-box; }
  #graphHeader input#nodeSearch::placeholder { color:#9ca3af; }
  #searchResults { position:absolute; top:calc(100% + 4px); left:0; right:0;
    background:rgba(28,28,28,0.95); backdrop-filter:blur(10px); -webkit-backdrop-filter:blur(10px);
    border:1px solid rgba(255,255,255,0.1); border-radius:8px; box-shadow:0 8px 24px rgba(0,0,0,0.5);
    max-height:280px; overflow-y:auto; z-index:1001; display:none; }
  .search-result { padding:6px 10px; cursor:pointer; color:#e5e7eb; font-size:12px; display:flex;
    justify-content:space-between; gap:8px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
  .search-result:hover, .search-result.active { background:rgba(56,189,248,0.18); }
  .search-result .sr-type { color:#9ca3af; font-size:10px; flex-shrink:0; }
</style>
<div id="graphBgGradient"></div>
<div id="graphHeader">
  <div style="font-size:19px; font-weight:700; color:#e5e7eb; letter-spacing:0.3px;
       text-shadow:0 2px 10px rgba(0,0,0,0.6); white-space:nowrap;">
    🎬 Grafo de Directores
  </div>
  <div id="graphStats" style="font-size:12px; color:#9ca3af; white-space:nowrap;"></div>
  <div id="searchWrap">
    <input type="text" id="nodeSearch" placeholder="Buscar director o película…" autocomplete="off" />
    <div id="searchResults"></div>
  </div>
</div>
"""

# Swatch de leyenda que reproduce el patrón de guiones real de cada tipo
# (mismos arrays que usa vis.js), para que la leyenda no se desincronice del grafo.
def _dash_swatch_style(hex_color, dashes, swatch_width=20):
    if not dashes:
        return (f"display:inline-block;width:{swatch_width}px;height:2px;"
                f"background:{hex_color};margin-right:6px;vertical-align:middle;")
    stops = []
    pos = 0
    on = True
    for seg in dashes:
        start = pos
        pos += seg
        stops.append(f"{hex_color if on else 'transparent'} {start}px {pos}px")
        on = not on
    total = pos
    gradient = ",".join(stops)
    return (f"display:inline-block;width:{swatch_width}px;height:2px;margin-right:6px;vertical-align:middle;"
            f"background-image:repeating-linear-gradient(to right,{gradient});background-size:{total}px 2px;")

_type_legend_rows = "\n".join(
    f'    <div><span style="{_dash_swatch_style(d["hex"], d["dashes"])}"></span>{d["name"]}</div>'
    for d in TYPE_STYLE_DEFS.values()
)
_default_legend_row = (
    f'    <div><span style="{_dash_swatch_style(DEFAULT_EDGE_STYLE_DEF["hex"], DEFAULT_EDGE_STYLE_DEF["dashes"])}"></span>'
    f'{DEFAULT_EDGE_STYLE_DEF["name"]}</div>'
)

_country_counts = (
    dirdf.loc[dirdf["references"] > 0, "country"]
    .fillna("Unknown")
    .value_counts()
)
_country_legend_rows = "\n".join(
    f'    <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;'
    f'background:{COUNTRY_COLORS.get(country, DEFAULT_COLOR)};margin-right:6px;vertical-align:middle;"></span>'
    f'{country} <span style="color:#9ca3af;">({count})</span></div>'
    for country, count in _country_counts.head(10).items()
)

legend_html = """
<div id="legendBox" style="position:fixed; bottom:14px; left:14px; background:rgba(34,34,34,0.72);
     backdrop-filter:blur(10px); -webkit-backdrop-filter:blur(10px);
     box-shadow:0 8px 28px rgba(0,0,0,0.45);
     color:#e5e7eb; font-size:12px; font-family:sans-serif; padding:10px 14px; border-radius:10px;
     border:1px solid rgba(255,255,255,0.08); z-index:1000; max-width:250px; line-height:1.5; max-height:calc(100vh - 28px); overflow-y:auto;">
  <div style="font-weight:bold; margin-bottom:6px; display:flex; justify-content:space-between; align-items:center;">
    Leyenda
    <span id="legendToggle" style="cursor:pointer; color:#9ca3af; font-weight:normal;">[ocultar]</span>
  </div>
  <div id="legendBody">
    <div style="color:#9ca3af; margin-bottom:2px;">Tipo de influencia</div>
__TYPE_LEGEND_ROWS__
__DEFAULT_LEGEND_ROW__
    <hr style="border-color:#444; margin:6px 0;">
    <div style="color:#9ca3af; margin-bottom:2px;">País del director (top 10)</div>
__COUNTRY_LEGEND_ROWS__
    <div style="color:#9ca3af; margin-top:4px; font-size:11px;">(el resto está en el panel de filtros →)</div>
  </div>
</div>
<script>
document.addEventListener("DOMContentLoaded", function () {
  var toggle = document.getElementById("legendToggle");
  var body = document.getElementById("legendBody");
  if (!toggle || !body) return;
  toggle.addEventListener("click", function () {
    var hidden = body.style.display === "none";
    body.style.display = hidden ? "block" : "none";
    toggle.textContent = hidden ? "[ocultar]" : "[mostrar]";
  });
});
</script>
"""
legend_html = legend_html.replace("__TYPE_LEGEND_ROWS__", _type_legend_rows)
legend_html = legend_html.replace("__DEFAULT_LEGEND_ROW__", _default_legend_row)
legend_html = legend_html.replace("__COUNTRY_LEGEND_ROWS__", _country_legend_rows)

_filter_tpl = """
<style>
#filterBox {position:fixed;top:10px;right:14px;background:rgba(34,34,34,0.72);
  backdrop-filter:blur(10px);-webkit-backdrop-filter:blur(10px);
  box-shadow:0 8px 28px rgba(0,0,0,0.45);
  color:#e5e7eb;font-size:12px;font-family:sans-serif;padding:10px 14px;
  border-radius:10px;border:1px solid rgba(255,255,255,0.08);z-index:1000;width:210px;
  max-height:calc(100vh - 20px);overflow-y:auto;}
#filterBox h3 {margin:0 0 6px;font-size:12px;font-weight:bold;
  display:flex;justify-content:space-between;align-items:center;}
#filterBox input[type=range] {width:100%;accent-color:#38bdf8;}
#countryList {max-height:180px;overflow-y:auto;margin-top:4px;}
#countryList label {display:flex;align-items:center;gap:6px;
  cursor:pointer;padding:2px 0;white-space:nowrap;overflow:hidden;}
.country-dot {width:10px;height:10px;border-radius:50%;flex-shrink:0;}
#filterReset {margin-top:8px;width:100%;padding:4px;background:#374151;
  color:#e5e7eb;border:1px solid #4b5563;border-radius:4px;cursor:pointer;}
#filterReset:hover {background:#4b5563;}
.cluster-btn {flex:1;padding:4px 2px;background:#374151;color:#e5e7eb;
  border:1px solid #4b5563;border-radius:4px;cursor:pointer;font-size:11px;}
.cluster-btn:hover {background:#4b5563;}
.cluster-btn.active {background:#1d4ed8;border-color:#60a5fa;}
.section-toggle {cursor:pointer;color:#9ca3af;font-size:10px;font-weight:normal;}
.mini-search {width:100%;padding:4px 8px;margin:4px 0;border-radius:6px;
  border:1px solid #4b5563;background:rgba(20,20,20,0.5);color:#e5e7eb;
  font-size:11px;box-sizing:border-box;}
.mini-search::placeholder {color:#6b7280;}
#resetAllBtn {margin-top:8px;width:100%;padding:5px;background:#374151;
  color:#e5e7eb;border:1px solid #4b5563;border-radius:4px;cursor:pointer;font-weight:500;}
#resetAllBtn:hover {background:#4b5563;}
</style>
<div id="filterBox">
  <h3>Filtros <span id="filterToggle" class="section-toggle">[ocultar]</span></h3>
  <div id="filterBody">
    <div style="margin-bottom:8px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:2px;">
        <span>Décadas</span>
        <span id="yearRangeLabel" style="color:#38bdf8;">YEAR_MIN_VAL–YEAR_MAX_VAL</span>
      </div>
      <input type="range" id="yearMin" min="YEAR_MIN_VAL" max="YEAR_MAX_VAL" value="YEAR_MIN_VAL" step="5">
      <input type="range" id="yearMax" min="YEAR_MIN_VAL" max="YEAR_MAX_VAL" value="YEAR_MAX_VAL" step="5">
    </div>
    <hr style="border-color:#444; margin:6px 0;">
    <button id="toggleMoviesBtn" class="cluster-btn" style="width:100%;margin-bottom:4px;">Ocultar películas</button>
    <hr style="border-color:#444; margin:6px 0;">
    <button id="toggleYearOrderBtn" class="cluster-btn" style="width:100%;margin-bottom:4px;">Ordenar por año</button>
    <hr style="border-color:#444; margin:6px 0;">
    <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:4px;">
      <span>Países</span>
      <span id="countryToggleAll" style="cursor:pointer;color:#9ca3af;">desmarcar todos</span>
    </div>
    <input type="text" id="countrySearchInput" class="mini-search" placeholder="Buscar país…" autocomplete="off">
    <div id="countryList"></div>
    <hr style="border-color:#444; margin:6px 0;">
    <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:4px;">
      <span>Géneros</span>
      <span id="genreToggleAll" style="cursor:pointer;color:#9ca3af;">desmarcar todos</span>
    </div>
    <input type="text" id="genreSearchInput" class="mini-search" placeholder="Buscar género…" autocomplete="off">
    <div id="genreList" style="max-height:130px;overflow-y:auto;"></div>
    <hr style="border-color:#444; margin:6px 0;">
    <div style="margin-bottom:4px;">Clustering</div>
    <div style="display:flex;gap:3px;">
      <button class="cluster-btn active" data-mode="none">Ninguno</button>
      <button class="cluster-btn" data-mode="country">País</button>
      <button class="cluster-btn" data-mode="genre">Género</button>
    </div>
    <hr style="border-color:#444; margin:6px 0;">
    <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:4px;">
      <span style="font-weight:500;">Física</span>
      <span id="physToggle" class="section-toggle">[ocultar]</span>
    </div>
    <div id="physBody">
      <div style="display:flex;justify-content:space-between;margin-bottom:1px;">
        <span>Repulsión</span><span id="physGravLbl" style="color:#38bdf8;">200</span>
      </div>
      <input type="range" id="physGrav" min="1" max="2000" value="200" step="10">
      <div style="display:flex;justify-content:space-between;margin-top:4px;margin-bottom:1px;">
        <span>Largo base</span><span id="physSLenLbl" style="color:#38bdf8;">200</span>
      </div>
      <input type="range" id="physSLen" min="10" max="3000" value="200" step="10">
      <div style="display:flex;justify-content:space-between;margin-top:4px;margin-bottom:1px;">
        <span>Rigidez</span><span id="physSConstLbl" style="color:#38bdf8;">0.18</span>
      </div>
      <input type="range" id="physSConst" min="1" max="300" value="18" step="1">
    </div>
    <hr style="border-color:#444; margin:6px 0;">
    <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:4px;">
      <span>Posiciones manuales</span>
      <span id="posSaveStatus" style="color:#4ade80;opacity:0;transition:opacity 0.3s;">Guardado ✓</span>
    </div>
    <div style="display:flex;gap:3px;margin-bottom:4px;">
      <button id="exportPosBtn" class="cluster-btn">Exportar</button>
      <button id="importPosBtn" class="cluster-btn">Importar</button>
    </div>
    <input type="file" id="importPosInput" accept="application/json" style="display:none;">
    <hr style="border-color:#444; margin:6px 0;">
    <button id="resetAllBtn">Restablecer todo</button>
  </div>
</div>
<script>
document.addEventListener("DOMContentLoaded", function () {
  ["filterToggle","physToggle"].forEach(function(id) {
    var toggle = document.getElementById(id);
    var bodyId = id === "filterToggle" ? "filterBody" : "physBody";
    var body = document.getElementById(bodyId);
    if (!toggle || !body) return;
    toggle.addEventListener("click", function () {
      var hidden = body.style.display === "none";
      body.style.display = hidden ? "block" : "none";
      toggle.textContent = hidden ? "[ocultar]" : "[mostrar]";
    });
  });
});
</script>
"""
filter_html = (_filter_tpl
    .replace("YEAR_MIN_VAL", str(YEAR_MIN))
    .replace("YEAR_MAX_VAL", str(YEAR_MAX)))

html = html.replace(
    '<div id="mynetwork"',
    header_html + legend_html + filter_html + '\n<div id="mynetwork"'
)

semantic_js = """
// ======= POSICIÓN INICIAL: eje Y según año (más antiguos arriba) =======
// Es solo el punto de partida: no queda "fixed", así que tanto la física
// como el arrastre manual del usuario la pueden mover libremente después.
// year <= 0 (o no numérico) significa "año desconocido": esos nodos quedan
// sin y inicial y la física los va acomodando cerca de sus vecinos con año.
// La escala no es lineal (potencia > 1): los años recientes, donde hay muchas
// más películas, quedan mucho más separados en Y que los años antiguos.
const YEAR_MIN = __YEAR_MIN_JS__, YEAR_MAX = __YEAR_MAX_JS__;
const YEAR_Y_TOP = -2000, YEAR_Y_BOTTOM = 8000;
const YEAR_SPREAD_EXPONENT = 2.3;
function yearToY(year) {
  const t = (Math.min(Math.max(year, YEAR_MIN), YEAR_MAX) - YEAR_MIN) / (YEAR_MAX - YEAR_MIN);
  return YEAR_Y_TOP + Math.pow(t, YEAR_SPREAD_EXPONENT) * (YEAR_Y_BOTTOM - YEAR_Y_TOP);
}
// Solo el director esta anclado por año: _yearY únicamente tiene entrada para
// nodos "director". Las películas no usan su propio año de estreno para nada
// - arrancan pegadas a su propio director, a corta distancia, más abajo.
const _yearY = {};
nodes.forEach(n => {
  if (n.node_type !== "director") return;
  const y = n.year;
  if (typeof y === "number" && !isNaN(y) && y > 0) _yearY[n.id] = yearToY(y);
});

// Director propio de cada película, vía la arista dtm (no cualquier arista
// dtm: una película solo tiene UN director real, aunque participe de varias
// relaciones de influencia).
const _nodeType = {};
nodes.forEach(n => { _nodeType[n.id] = n.node_type; });
const _movieOwnDirectors = {};
edges.forEach(e => {
  const ft = _nodeType[e.from], tt = _nodeType[e.to];
  if (ft === "director" && tt === "movie") {
    (_movieOwnDirectors[e.to] = _movieOwnDirectors[e.to] || []).push(e.from);
  } else if (ft === "movie" && tt === "director") {
    (_movieOwnDirectors[e.from] = _movieOwnDirectors[e.from] || []).push(e.to);
  }
});

// Posición inicial de cada película: un punto a corta distancia (radio chico
// y aleatorio, para que no queden todas exactamente superpuestas) alrededor
// de la posición actual de su propio director - nunca su propio año de
// estreno. Antes, un director con carrera larga (debut en los 60, películas
// hasta los 2020) terminaba con sus propias películas a miles de píxeles de
// distancia en Y, porque el año de estreno se dispara mucho más en la escala
// no lineal que el año de debut del director. _initialPositions también se
// usa en "Restablecer posiciones" para volver a este mismo punto de partida.
// 260 en vez de 150: con las peliculas mas chicas (MOVIE_SIZE_SCALE) quedaban
// demasiado pegadas al director - un poco mas de aire entre ellas y su
// director sin dejar de leerse como "su" filmografia agrupada.
const MOVIE_SPREAD_RADIUS = 260;
const _initialPositions = {};
nodes.forEach(n => {
  if (n.node_type === "director" && _yearY[n.id] !== undefined) {
    _initialPositions[n.id] = { y: _yearY[n.id] };
  }
});
nodes.forEach(n => {
  if (n.node_type !== "movie") return;
  const dirId = (_movieOwnDirectors[n.id] || []).find(d => _yearY[d] !== undefined);
  if (dirId === undefined) return;
  const dirPos = network.getPositions([dirId])[dirId] || { x: 0 };
  const angle = Math.random() * Math.PI * 2;
  const radius = Math.random() * MOVIE_SPREAD_RADIUS;
  _initialPositions[n.id] = {
    x: dirPos.x + Math.cos(angle) * radius,
    y: _yearY[dirId] + Math.sin(angle) * radius,
  };
});
nodes.update(Object.keys(_initialPositions).map(id => ({ id, ..._initialPositions[id] })));

// ======================= ANCLAJE PELICULA -> DIRECTOR =======================
// Con ~1300 nodos la repulsión general del solver le gana a cualquier resorte
// razonable: aun arrancando a 150px de su director, la física terminaba
// empujando cada película a ~3000-3500px de distancia otra vez. En vez de
// pelear contra la repulsión, las películas dejan de participar de la física
// del todo (siempre physics:false en la vista global) y se anclan a mano a la
// posición ACTUAL de su propio director, con el mismo offset corto calculado
// una sola vez al cargar - así quedan agrupadas de verdad, y de paso bajan
// muchísimo el costo de la estabilización (el solver ya no tiene que mover
// ~850 nodos más).
const _movieOffset = {};
Object.keys(_initialPositions).forEach(id => {
  const n = nodes.get(id);
  if (!n || n.node_type !== "movie") return;
  const dirId = (_movieOwnDirectors[id] || []).find(d => _yearY[d] !== undefined);
  if (dirId === undefined) return;
  const dirPos0 = network.getPositions([dirId])[dirId] || { x: 0, y: _yearY[dirId] };
  const moviePos0 = _initialPositions[id];
  _movieOffset[id] = { dx: moviePos0.x - dirPos0.x, dy: moviePos0.y - dirPos0.y, dirId };
});

function syncMoviePositions() {
  const dirIds = [...new Set(Object.values(_movieOffset).map(o => o.dirId))];
  const dirPos = network.getPositions(dirIds);
  const updates = [];
  Object.keys(_movieOffset).forEach(movieId => {
    const n = nodes.get(movieId);
    if (!n || n.hidden) return;
    const { dx, dy, dirId } = _movieOffset[movieId];
    const dp = dirPos[dirId];
    if (!dp) return;
    // Con "Ordenar por año" activo, la pelicula usa su PROPIO año de estreno
    // para la fila (no el de debut de su director) - un director con carrera
    // larga tiene peliculas repartidas en varias filas, no todas pegadas a
    // la fila de su debut.
    const y = _yearOrderActive
      ? yearToY((typeof n.year === "number" && !isNaN(n.year) && n.year > 0) ? n.year : YEAR_ORDER_DEFAULT)
      : dp.y + dy;
    updates.push({ id: movieId, x: dp.x + dx, y, physics: false });
  });
  if (updates.length) nodes.update(updates);
}


// ======================= POSICIONES MANUALES (drag) =======================
// Todo nodo que el usuario arrastra queda excluido de la física para siempre
// (physics:false, no "fixed" — así sigue arrastrable libremente en cualquier
// eje, cuantas veces se quiera). Se guarda en localStorage (cada archivo
// file:// tiene su propio origen, así que persiste entre recargas de este
// mismo HTML) y tiene prioridad sobre la física en applyFilters() y
// highlightSemantic().
const POS_STORAGE_KEY = "grafo_directors_manual_positions_v1";
let _manualPos = {};
try { _manualPos = JSON.parse(localStorage.getItem(POS_STORAGE_KEY) || "{}"); } catch (e) { _manualPos = {}; }
const _manualFixedIds = new Set(Object.keys(_manualPos));

function saveManualPositions() {
  try { localStorage.setItem(POS_STORAGE_KEY, JSON.stringify(_manualPos)); } catch (e) {}
}

function showPosSavedIndicator() {
  const el = document.getElementById("posSaveStatus");
  if (!el) return;
  el.textContent = "Guardado ✓";
  el.style.opacity = 1;
  clearTimeout(el._hideTimer);
  el._hideTimer = setTimeout(() => { el.style.opacity = 0; }, 1500);
}

function applyManualPositions() {
  const updates = Object.entries(_manualPos)
    .filter(([id]) => nodes.get(id))
    .map(([id, pos]) => ({ id, x: pos.x, y: pos.y, physics: false }));
  if (updates.length) nodes.update(updates);
}
applyManualPositions();

network.on("dragEnd", function (params) {
  if (!params.nodes || !params.nodes.length) return;
  const positions = network.getPositions(params.nodes);
  params.nodes.forEach(id => {
    if (isHubNode(nodes.get(id))) return;
    _manualPos[id] = { x: positions[id].x, y: positions[id].y };
    _manualFixedIds.add(id);
    nodes.update({ id, physics: false });
  });
  saveManualPositions();
  showPosSavedIndicator();
  if (_highlightSet === null && _clusterMode !== "genre") syncMoviePositions();
});

function exportManualPositions() {
  const blob = new Blob([JSON.stringify(_manualPos, null, 2)], { type: "application/json" });
  const a = document.createElement("a");
  a.href = URL.createObjectURL(blob);
  a.download = "posiciones_grafo.json";
  a.click();
  URL.revokeObjectURL(a.href);
}

function importManualPositions(file) {
  const reader = new FileReader();
  reader.onload = () => {
    try {
      const parsed = JSON.parse(reader.result);
      Object.entries(parsed).forEach(([id, pos]) => {
        _manualPos[id] = pos;
        _manualFixedIds.add(id);
      });
      saveManualPositions();
      applyManualPositions();
    } catch (e) { alert("Archivo de posiciones inválido"); }
  };
  reader.readAsText(file);
}

function resetManualPositions() {
  // Vuelve a la posición inicial calculada arriba: el director según su año,
  // cada película a corta distancia de su propio director — no solo los
  // nodos que se habían movido a mano, para que "restablecer" reordene todo.
  const updates = Object.keys(_initialPositions).map(id => ({ id, ..._initialPositions[id] }));
  _manualPos = {};
  _manualFixedIds.clear();
  saveManualPositions();
  if (updates.length) nodes.update(updates);
  applyFilters();
}

// ======================= SEARCH =======================
function searchNodeByLabel(query) {
  if (!query) return;
  query = query.toLowerCase();
  const allNodes = nodes.get();
  for (let i = 0; i < allNodes.length; i++) {
    const n = allNodes[i];
    if ((n.label || "").toLowerCase().includes(query)) {
      network.selectNodes([n.id]);
      highlightSemantic(n.id);
      network.focus(n.id, { scale: 1.5, animation: { duration: 800, easingFunction: "easeInOutQuad" } });
      return;
    }
  }
}

// ============== DIRECTIONAL NEIGHBORHOOD ==============
// Devuelve Map(nodeId -> distancia en saltos desde startId): usado tanto
// para decidir qué queda visible como para acomodar arriba/abajo según qué
// tan lejos influye o es influenciado cada nodo.
// network.getConnectedNodes() recorre TODAS las aristas del dataset, incluidas
// las virtuales director->director (__vdir_, ver _virtualDirEdges): siguen
// existiendo estructuralmente aunque esten pintadas invisibles, asi que un
// recorrido con esa funcion "salta" director->director sin pasar por
// peliculas reales. getDirectedNeighbors() replica la misma semantica de
// direccion pero solo sobre aristas reales, no-hub.
function getDirectedNeighbors(nodeId, direction) {
  const result = [];
  edges.forEach(e => {
    if (isHubEdge(e)) return;
    if (typeof e.id === "string" && e.id.startsWith("__vdir_")) return;
    if (direction === "to" && e.from === nodeId) result.push(e.to);
    else if (direction === "from" && e.to === nodeId) result.push(e.from);
  });
  return result;
}

function getDirectionalNeighborhood(startId, direction) {
  // anchor = el vecino de grado 1 (vecino directo del nodo elegido) por el
  // que se llegó a este nodo. Se usa después para agrupar visualmente los
  // nodos de grado 2 y 3 alrededor de su nodo de grado 1, en vez de dejarlos
  // sueltos en la misma franja que el resto de la vecindad.
  const info = new Map([[startId, { depth: 0, anchor: startId }]]);
  const startNode = nodes.get(startId);
  if (!startNode) return info;
  getDirectedNeighbors(startId, direction).forEach(n1 => {
    const node1 = nodes.get(n1);
    if (!node1 || (node1.node_type !== "movie" && node1.node_type !== "director")) return;
    if (!info.has(n1)) info.set(n1, { depth: 1, anchor: n1 });
    if (node1.node_type === "director" && n1 !== startId) return;
    if (node1.node_type === "movie") {
      getDirectedNeighbors(n1, direction).forEach(n2 => {
        const node2 = nodes.get(n2);
        if (!node2 || (node2.node_type !== "movie" && node2.node_type !== "director")) return;
        if (!info.has(n2)) info.set(n2, { depth: 2, anchor: n1 });
        if (node2.node_type === "director") return;
        if (node2.node_type === "movie") {
          getDirectedNeighbors(n2, direction).forEach(n3 => {
            const node3 = nodes.get(n3);
            if (node3 && node3.node_type === "director" && !info.has(n3)) info.set(n3, { depth: 3, anchor: n1 });
          });
        }
      });
    }
  });
  return info;
}

// ======================= ETIQUETAS SEGÚN ZOOM =======================
const LABEL_ZOOM_THRESHOLD = 0.04;
const nodeBaseFontSize = {};
nodes.forEach(n => { nodeBaseFontSize[n.id] = 20; });
let _labelRefreshTimer = null;
let _lastLabelState = null;
function refreshLabelSizes() {
  // Con un highlight activo (nodo seleccionado) el título siempre tiene que
  // leerse, sin importar a qué escala termine el fit() del sub-grafo: es una
  // vista chica y curada, no el grafo completo.
  const showLabels = _highlightSet !== null || network.getScale() >= LABEL_ZOOM_THRESHOLD;
  if (showLabels === _lastLabelState) return;
  _lastLabelState = showLabels;
  const updates = [];
  nodes.forEach(n => {
    if (n.hidden) return;
    updates.push({ id: n.id, font: { size: showLabels ? (nodeBaseFontSize[n.id] || 20) : 0 } });
  });
  nodes.update(updates);
}
network.on("zoom", function () {
  clearTimeout(_labelRefreshTimer);
  _labelRefreshTimer = setTimeout(refreshLabelSizes, 120);
});

// Apagar la simulacion fisica apenas se estabiliza la primera vez: sin esto
// queda corriendo indefinidamente y los nodos siguen alejandose de a poco.
// pyvis vuelve a mostrar la barra de carga en cada stabilize() posterior
// (ver runBoundedPhysics). Esta regla, con !important, gana siempre sobre
// los removeAttribute("style") que hace pyvis, así que una vez que la
// barra recibe la clase "graphReady" (la primera vez que termina de
// cargar) queda oculta para siempre sin importar cuántas veces se
// vuelva a correr la física.
(function () {
  const style = document.createElement("style");
  style.textContent = "#loadingBar.graphReady { display: none !important; opacity: 0 !important; }";
  document.head.appendChild(style);
})();

// ======================= ESTADO GLOBAL =======================
const COUNTRY_COLORS = __COUNTRY_COLORS_JSON__;
const TYPE_STYLES = __TYPE_STYLES_JSON__;
const DEFAULT_EDGE_STYLE = __DEFAULT_EDGE_STYLE_JSON__;
let _filterCountries = new Set();
let _filterGenres    = new Set();
let _filterYearMin   = null;
let _filterYearMax   = null;
let _hideMovies      = true;
let _highlightSet    = null;
let _physCurrent     = { grav: -200, slen: 200, sconst: 0.18 };

// ======================= STATS EN VIVO (header) =======================
function updateHeaderStats() {
  const el = document.getElementById("graphStats");
  if (!el) return;
  let visDirs = 0, totalDirs = 0, visMovies = 0, totalMovies = 0;
  nodes.forEach(n => {
    if (n.node_type === "director") { totalDirs++; if (!n.hidden) visDirs++; }
    else if (n.node_type === "movie") { totalMovies++; if (!n.hidden) visMovies++; }
  });
  el.textContent = `${visDirs}/${totalDirs} directores · ${visMovies}/${totalMovies} películas`;
}

// Orden fijo de tipos (mismo orden que TYPE_STYLES) usado para desempatar
// cuál tipo "gana" cuando varias rutas director→película→película→director
// colapsan en la misma arista virtual director→director.
const TYPE_RANK = {};
Object.keys(TYPE_STYLES).forEach((k, i) => { TYPE_RANK[k] = i; });
function typeRankOf(t) { return (t && t in TYPE_RANK) ? TYPE_RANK[t] : 99; }

// Parámetros de física compacta (usados en highlight y filtros)
// springLength=80 (mucho mas chico que los ~140-260px que ya calcula
// layoutBranch para separar hermanos/niveles) hacia que esta fisica
// COMPRIMIERA el arbol prolijo en vez de solo pulirlo - quedaba apretado y
// sin movimiento visible real. Con springLength mas parecido a esos gaps y
// algo menos de rigidez/damping, se nota el asentado sin aplastar el layout.
const COMPACT_PHYS = { gravitationalConstant: -220, springLength: 160, springConstant: 0.1, centralGravity: 0.02, damping: 0.3 };

// Física dedicada para el modo de clustering "País": la global (_physCurrent)
// es demasiado floja para juntar countries grandes (USA tiene 215 directores
// en un solo hub) - medido, con la física global el promedio real terminaba
// a ~1850px del hub (vs. los 120px que pide la arista). Con esta física mucho
// más agresiva baja a ~400-550px, que es cerca del mínimo físico dado el
// tamaño de los nodos (no hay forma de juntar 215 nodos de ese tamaño mucho
// más sin que se superpongan).
const COUNTRY_CLUSTER_PHYS = { gravitationalConstant: -15, springLength: 60, springConstant: 1.0, centralGravity: 0.02, damping: 0.6, avoidOverlap: 0.8 };

// IMPORTANTE: vis.js no hace merge de "enabled" entre llamadas a setOptions -
// si esta llamada no especifica "enabled", lo resetea a su default (true),
// aunque un momento antes se hubiera apagado explicitamente con
// network.setOptions({physics:{enabled:false}}). Sin este "enabled: false"
// ACA, cada highlight o cambio de filtro dejaba la simulacion corriendo de
// fondo para siempre (girando/oscilando sin parar nunca), porque
// restorePhysics() es justamente lo ultimo que se llama despues de apagar
// la fisica en runBoundedPhysics(). Quien necesita prenderla de nuevo
// (runBoundedPhysics) ya lo hace explicito con su propio setOptions.
function restorePhysics() {
  network.setOptions({ physics: { enabled: false, forceAtlas2Based: {
    gravitationalConstant: _physCurrent.grav,
    springLength: _physCurrent.slen,
    springConstant: _physCurrent.sconst,
    centralGravity: 0.01,
    damping: 0.4,
    avoidOverlap: 0.8,
  }}});
}

let _stabilizeSeq = 0;

// Corre la física hasta que se estabilice (o como máximo hasta el timeout
// de seguridad) y la apaga. Se fuerza network.stabilize() en vez de solo
// esperar "stabilizationIterationsDone" al reactivar la física: ese evento
// no dispara de forma confiable en reactivaciones posteriores a la
// primera, así que sin esto la simulación queda corriendo para siempre
// (el grafo se expande sin parar) apenas se selecciona o deselecciona un
// nodo, se cambia un filtro, etc.
function runBoundedPhysics(afterFn, timeoutMs, iterations) {
  const seq = ++_stabilizeSeq;
  let done = false;
  function finish() {
    if (done || seq !== _stabilizeSeq) return;
    done = true;
    network.setOptions({ physics: { enabled: false } });
    // network.stabilize() reactiva el listener de pyvis que muestra la
    // barra de carga ("stabilizationProgress"), pero el que la oculta
    // ("stabilizationIterationsDone") es de un solo uso y ya se gastó en
    // la carga inicial — sin esto, la barra queda pegada visible después
    // del primer click en un nodo.
    const loadingBar = document.getElementById("loadingBar");
    if (loadingBar) {
      loadingBar.style.opacity = 0;
      loadingBar.style.display = "none";
      loadingBar.classList.add("graphReady");
    }
    if (afterFn) afterFn();
  }
  network.once("stabilizationIterationsDone", finish);
  network.setOptions({ physics: { enabled: true } });
  // Un reacomodo interactivo (seleccionar/deseleccionar, filtrar) no
  // necesita redesempacar el grafo entero: con pocas iteraciones alcanza
  // y corre mucho más rápido que repetir la pasada inicial completa.
  network.stabilize(iterations || 150);
  setTimeout(finish, timeoutMs || 4000);
}

function applyCompactPhysics(afterFn) {
  // A diferencia de runBoundedPhysics() (que usa network.stabilize(), un
  // salto directo al resultado sin animar nada intermedio), acá physics
  // corre "en vivo": se apaga stabilization para que activar physics
  // simule cuadro a cuadro y se vea de verdad como el desorden se acomoda
  // en tiempo real, no un salto instantáneo.
  const seq = ++_stabilizeSeq;
  let done = false;
  function finish() {
    if (done || seq !== _stabilizeSeq) return;
    done = true;
    network.setOptions({ physics: { enabled: false, stabilization: true } });
    restorePhysics();
    if (afterFn) afterFn();
  }
  network.setOptions({ physics: { enabled: false, stabilization: false, forceAtlas2Based: COMPACT_PHYS } });
  network.once("stabilized", finish);
  network.setOptions({ physics: { enabled: true } });
  setTimeout(finish, 3000);
}

function isHubNode(n) {
  return n && (n.node_type === "country_hub" || n.node_type === "genre_hub");
}

function isHubEdge(e) {
  const t = nodes.get(e.to), f = nodes.get(e.from);
  return isHubNode(t) || isHubNode(f);
}

function hubNodeOfEdge(e) {
  const t = nodes.get(e.to), f = nodes.get(e.from);
  return isHubNode(t) ? t : isHubNode(f) ? f : null;
}

// ============ VIRTUAL DIRECTOR→DIRECTOR EDGES (para modo sin películas) ============
const _virtualDirEdges = (function () {
  const nType = {};
  nodes.forEach(n => { nType[n.id] = n.node_type; });

  const dirToMovies = {}, movieToMovies = {}, movieToDirs = {};
  // Todas las aristas reales (no-hub) agrupadas por par de nodos SIN ORDEN:
  // cualquier par de nodos con más de una arista entre sí (incluso en
  // sentidos opuestos, o una real + una virtual director→director) se
  // dibuja exactamente superpuesta —y solo se ve una flecha— si no se curva.
  const pairGroups = {};
  edges.forEach(e => {
    if (isHubEdge(e)) return;
    const ft = nType[e.from], tt = nType[e.to];
    if (ft === "director" && tt === "movie") {
      (dirToMovies[e.from] = dirToMovies[e.from] || []).push({ movie: e.to, influence_type: e.influence_type });
    } else if (ft === "movie" && tt === "movie") {
      (movieToMovies[e.from] = movieToMovies[e.from] || []).push(e.to);
    } else if (ft === "movie" && tt === "director") {
      (movieToDirs[e.from] = movieToDirs[e.from] || []).push({ dir: e.to, influence_type: e.influence_type });
    }
    const ukey = [e.from, e.to].sort().join("\x00");
    (pairGroups[ukey] = pairGroups[ukey] || []).push({ real: true, id: e.id });
  });

  // Cuando varias rutas director→película→película→director colapsan en el
  // mismo par director→director, se queda con el tipo de influencia más
  // "declarativo" (menor rank) de todas las rutas encontradas.
  const pairType = {};
  Object.entries(dirToMovies).forEach(([src, smList]) => {
    smList.forEach(({ movie: sm, influence_type: t1 }) => {
      (movieToMovies[sm] || []).forEach(tm => {
        (movieToDirs[tm] || []).forEach(({ dir: tgt, influence_type: t2 }) => {
          if (src === tgt) return;
          const key = src + "\x00" + tgt;
          const t = typeRankOf(t1) <= typeRankOf(t2) ? t1 : t2;
          if (!(key in pairType) || typeRankOf(t) < typeRankOf(pairType[key])) pairType[key] = t;
        });
      });
    });
  });

  // Las aristas virtuales director→director se suman al mismo pairGroups
  // (por par SIN ORDEN, a diferencia de pairType que sí distingue sentido)
  // para curvarse junto con cualquier arista real que ya exista entre ese
  // mismo par.
  const virtualEdges = Object.entries(pairType).map(([key, t], i) => {
    const sep = key.indexOf("\x00");
    const style = TYPE_STYLES[t] || DEFAULT_EDGE_STYLE;
    const from = key.slice(0, sep), to = key.slice(sep + 1);
    const edgeObj = {
      id: "__vdir_" + i,
      from, to,
      influence_type: t,
      length: 1280,
      base_color: style.color,
      base_width: style.width,
      width: style.width,
      base_dashes: style.dashes,
      base_physics: true,
      color: "rgba(0,0,0,0)",
      physics: false,
    };
    const ukey = [from, to].sort().join("\x00");
    (pairGroups[ukey] = pairGroups[ukey] || []).push({ real: false, id: edgeObj.id, edgeObj });
    return edgeObj;
  });

  const realCurveUpdates = [];
  Object.values(pairGroups).forEach(group => {
    if (group.length <= 1) return;
    group.forEach((item, idx) => {
      const side = idx % 2 === 0 ? "curvedCW" : "curvedCCW";
      const roundness = 0.15 + 0.12 * Math.floor(idx / 2);
      const smooth = { enabled: true, type: side, roundness };
      if (item.real) {
        realCurveUpdates.push({ id: item.id, smooth });
      } else {
        item.edgeObj.smooth = smooth;
      }
    });
  });
  if (realCurveUpdates.length) edges.update(realCurveUpdates);

  return virtualEdges;
})();

function passesFilter(n, opts) {
  if (!n || isHubNode(n)) return true;
  if (n.node_type === "movie" && _hideMovies && !(opts && opts.ignoreHideMovies)) return false;
  if (n.node_type === "director") {
    if (_filterCountries.size > 0 && _filterCountries.has(n.country)) return false;
    if (_filterYearMin !== null && n.year !== null && n.year < _filterYearMin) return false;
    if (_filterYearMax !== null && n.year !== null && n.year > _filterYearMax) return false;
  }
  if (n.node_type === "movie" && _filterGenres.size > 0) {
    const mg = (n.genres || "").split(",").map(g => g.trim());
    if (!mg.some(g => !_filterGenres.has(g))) return false;
  }
  return true;
}

function baseEdgePhysics(e) {
  const hub = hubNodeOfEdge(e);
  if (hub) {
    return (hub.node_type === "country_hub" && _clusterMode === "country")
        || (hub.node_type === "genre_hub"   && _clusterMode === "genre");
  }
  // Fuera de esto, una arista de relacion real (no-hub) compite con la
  // atraccion (debil) hacia el hub de pais/genero: con el spring reforzado
  // de COUNTRY_CLUSTER_PHYS termina venciendola y el grafo colapsa en un
  // solo bloque en vez de separarse. Se apagan mientras haya un cluster activo.
  if (_clusterMode === "country" || _clusterMode === "genre") return false;
  return e.base_physics !== false;
}

// Nodos que un filtro activo (país/género/año/ocultar películas) excluye.
// Se comparte entre applyFilters y highlightSemantic: un nodo filtrado no
// debe reaparecer solo porque el highlight lo incluyó en su recorrido.
function computeHiddenByFilter(opts) {
  const hiddenByFilter = new Set();
  nodes.forEach(n => { if (!passesFilter(n, opts)) hiddenByFilter.add(n.id); });

  // Películas excluidas específicamente por el filtro de género, sin importar
  // _hideMovies: se usa más abajo para ocultar también a los directores cuyas
  // películas quedan todas fuera del género, incluso en modo "ocultar
  // películas" (donde passesFilter ya oculta TODAS las películas y por eso
  // no sirve para distinguir cuáles caen por género).
  const genreHiddenMovies = new Set();
  if (_filterGenres.size > 0) {
    nodes.forEach(n => {
      if (n.node_type !== "movie") return;
      const mg = (n.genres || "").split(",").map(g => g.trim());
      if (!mg.some(g => !_filterGenres.has(g))) genreHiddenMovies.add(n.id);
    });
  }

  nodes.forEach(n => {
    if (n.node_type !== "movie") return;
    const dirs = network.getConnectedNodes(n.id).filter(id => {
      const nd = nodes.get(id); return nd && nd.node_type === "director";
    });
    if (dirs.length > 0 && dirs.every(id => hiddenByFilter.has(id))) hiddenByFilter.add(n.id);
  });
  if (_filterGenres.size > 0) {
    nodes.forEach(n => {
      if (n.node_type !== "director" || hiddenByFilter.has(n.id)) return;
      const movies = network.getConnectedNodes(n.id).filter(id => {
        const nd = nodes.get(id); return nd && nd.node_type === "movie";
      });
      if (movies.length > 0 && movies.every(id => genreHiddenMovies.has(id))) hiddenByFilter.add(n.id);
    });
  }
  return hiddenByFilter;
}

function applyFilters() {
  const hiddenByFilter = computeHiddenByFilter();
  // Las posiciones movidas a mano solo se respetan en la vista global (sin
  // clustering ni highlight activo); al clusterizar o resaltar un nodo se
  // liberan como cualquier otro nodo, y vuelven a su lugar guardado al volver
  // a la vista global (p.ej. al deseleccionar).
  const inGlobalView = _clusterMode === "none" && _highlightSet === null;
  const nodeUpdates = [];
  nodes.forEach(n => {
    if (isHubNode(n)) return;
    const filteredOut = hiddenByFilter.has(n.id);
    const hidden = _highlightSet !== null ? (!_highlightSet.has(n.id) || filteredOut) : filteredOut;
    nodeBaseFontSize[n.id] = 20;
    const isManual = _manualFixedIds.has(n.id);
    const update = { id: n.id, hidden };
    if (hidden) {
      update.physics = false;
    } else if (_highlightSet === null && n.node_type === "movie") {
      // Las películas solo tienen física propia en el modo de clustering
      // "Género" (para responder a los hubs de género); en cualquier otro
      // modo quedan ancladas a su director (ver syncMoviePositions más abajo).
      update.physics = _clusterMode === "genre";
    } else if (inGlobalView && isManual) {
      update.physics = false;
      update.x = _manualPos[n.id].x;
      update.y = _manualPos[n.id].y;
    } else {
      update.physics = true;
    }
    nodeUpdates.push(update);
  });
  nodes.update(nodeUpdates);
  if (_highlightSet === null && _clusterMode !== "genre") syncMoviePositions();
  updateHeaderStats();
  const edgeUpdates = [];
  edges.forEach(e => {
    const filteredOut = hiddenByFilter.has(e.from) || hiddenByFilter.has(e.to);
    // Con un highlight activo, una arista además tiene que seguir dentro del
    // mini-grafo resaltado (y las virtuales director→director siguen sin
    // mostrarse nunca en highlight) — si no, cambiar un filtro mientras hay
    // un nodo destacado hacía reaparecer cualquier arista no filtrada.
    const isVirtual = typeof e.id === "string" && e.id.startsWith("__vdir_");
    const outOfHighlight = _highlightSet !== null
      && (isVirtual || !_highlightSet.has(e.from) || !_highlightSet.has(e.to));
    const hide = filteredOut || outOfHighlight;
    const physOn = hide ? false : baseEdgePhysics(e);
    edgeUpdates.push({
      id: e.id,
      color: hide ? "rgba(0,0,0,0)" : (e.base_color || DEFAULT_EDGE_STYLE.color),
      dashes: hide ? false : (e.base_dashes || false),
      physics: physOn,
    });
  });
  edges.update(edgeUpdates);
  refreshLabelSizes();

  restorePhysics();
  runBoundedPhysics(() => {
    if (_highlightSet === null && _clusterMode !== "genre") syncMoviePositions();
    network.fit({ animation: { duration: 400, easingFunction: "easeInOutQuad" } });
  });
}

// ======================= CLUSTERING =======================
let _clusterMode = "none";

function setClusterMode(mode) {
  _clusterMode = mode;
  // Género agrupa películas: si estaban ocultas (la vista global las
  // esconde por defecto) no habría nada que agrupar - se muestran solas.
  // applyFilters() es necesario acá (no solo cambiar _hideMovies): es lo que
  // recalcula la propiedad "hidden" real de cada nodo película.
  if (mode === "genre" && _hideMovies) {
    _hideMovies = false;
    const tmBtn = document.getElementById("toggleMoviesBtn");
    if (tmBtn) { tmBtn.textContent = "Ocultar películas"; tmBtn.classList.remove("active"); }
    edges.remove(_virtualDirEdges.map(e => e.id));
    applyFilters();
  }
  const nodeUpdates = [], edgeUpdates = [];
  // Igual que en applyFilters(): las posiciones movidas a mano solo se
  // respetan en la vista global. Al clusterizar se liberan; al volver a
  // "Ninguno" (si tampoco hay highlight activo) se restauran a su lugar.
  const inGlobalView = mode === "none" && _highlightSet === null;
  nodes.forEach(n => {
    if (isHubNode(n)) {
      const active = (n.node_type === "country_hub" && mode === "country")
                  || (n.node_type === "genre_hub"   && mode === "genre");
      nodeUpdates.push({ id: n.id, physics: active });
    } else if (n.node_type === "movie") {
      // Las películas solo tienen física propia en modo "Género" (para
      // responder a los hubs de género); fuera de ese modo quedan ancladas
      // a su director (ver syncMoviePositions en applyFilters()).
      if (!n.hidden) nodeUpdates.push({ id: n.id, physics: mode === "genre" });
    } else if (_manualFixedIds.has(n.id)) {
      nodeUpdates.push(inGlobalView
        ? { id: n.id, x: _manualPos[n.id].x, y: _manualPos[n.id].y, physics: false }
        : { id: n.id, physics: true });
    }
  });
  nodes.update(nodeUpdates);
  edges.forEach(e => {
    const hub = hubNodeOfEdge(e);
    if (hub) {
      const active = (hub.node_type === "country_hub" && mode === "country")
                  || (hub.node_type === "genre_hub"   && mode === "genre");
      edgeUpdates.push({ id: e.id, physics: active });
      return;
    }
    if (mode === "country" || mode === "genre") {
      edgeUpdates.push({ id: e.id, physics: false });
    }
  });
  edges.update(edgeUpdates);
  document.querySelectorAll(".cluster-btn[data-mode]").forEach(btn => {
    btn.classList.toggle("active", btn.dataset.mode === mode);
  });
  // Encender physics en los nodos/aristas del hub no alcanza si el solver ya
  // se había "asentado" — stabilize() puede terminar casi enseguida sin
  // notar la fuerza nueva. Se fuerza un reinicio real (disable + reconfigurar
  // parámetros) y se dan más iteraciones: reacomodar cientos de directores
  // en clusters es una reorganización grande, no un ajuste liviano como un
  // highlight.
  const isCountryMode = mode === "country";
  const isGenreMode = mode === "genre";
  const clusterPhys = (isCountryMode || isGenreMode)
    ? COUNTRY_CLUSTER_PHYS
    : {
        gravitationalConstant: _physCurrent.grav,
        springLength: _physCurrent.slen,
        springConstant: _physCurrent.sconst,
        centralGravity: 0.01,
        damping: 0.4,
        avoidOverlap: 0.8,
      };
  network.setOptions({ physics: { enabled: false, forceAtlas2Based: clusterPhys } });
  runBoundedPhysics(() => {
    // Al salir de "Género" las películas vuelven a anclarse a su director;
    // en "Género" se dejan donde las asentó la física de los hubs.
    if (mode !== "genre" && _highlightSet === null) syncMoviePositions();
    updateHeaderStats();
  }, 8000, (isCountryMode || isGenreMode) ? 800 : 400);
}

// Solo los botones con data-mode son de clustering — toggleMoviesBtn
// comparte la clase "cluster-btn" únicamente por estilo visual y no debe
// disparar setClusterMode.
document.querySelectorAll(".cluster-btn[data-mode]").forEach(btn => {
  btn.addEventListener("click", () => setClusterMode(btn.dataset.mode));
});
setClusterMode("none");
// Garantizar que ninguna película quede fija al cargar
nodes.update(nodes.get({ filter: n => n.node_type === "movie" })
  .map(n => ({ id: n.id, fixed: { x: false, y: false } })));

// ======================= FÍSICA SLIDERS =======================
(function() {
  const gravS  = document.getElementById("physGrav");
  const slenS  = document.getElementById("physSLen");
  const sconstS = document.getElementById("physSConst");
  if (!gravS || !slenS || !sconstS) return;
  let _physDebounce = null;
  function applyPhysics() {
    const grav   = -parseInt(gravS.value);
    const slen   = parseInt(slenS.value);
    const sconst = parseInt(sconstS.value) / 100;
    _physCurrent = { grav, slen, sconst };
    document.getElementById("physGravLbl").textContent  = gravS.value;
    document.getElementById("physSLenLbl").textContent  = slen;
    document.getElementById("physSConstLbl").textContent = sconst.toFixed(2);
    restorePhysics();
    clearTimeout(_physDebounce);
    _physDebounce = setTimeout(function() {
      runBoundedPhysics();
    }, 400);
  }
  gravS.addEventListener("input",   applyPhysics);
  slenS.addEventListener("input",   applyPhysics);
  sconstS.addEventListener("input", applyPhysics);
})();

// ======================= COUNTRY / GENRE CHECKBOXES =======================
(function buildCountryFilter() {
  const countryCount = {};
  nodes.forEach(n => {
    if (n.node_type === "director" && n.country)
      countryCount[n.country] = (countryCount[n.country] || 0) + 1;
  });
  const sorted = Object.entries(countryCount).sort((a,b) => b[1]-a[1]);
  const list = document.getElementById("countryList");
  if (!list) return;
  sorted.forEach(([country, count]) => {
    const color = COUNTRY_COLORS[country] || "#9ca3af";
    const lbl = document.createElement("label");
    lbl.title = count + " directores";
    lbl.innerHTML = '<input type="checkbox" checked data-country="' + country + '">'
      + '<span class="country-dot" style="background:' + color + '"></span>'
      + '<span style="overflow:hidden;text-overflow:ellipsis;">' + country
      + ' <span style="color:#9ca3af;">(' + count + ')</span></span>';
    list.appendChild(lbl);
    lbl.querySelector("input").addEventListener("change", function () {
      if (this.checked) _filterCountries.delete(country);
      else _filterCountries.add(country);
      applyFilters();
    });
  });
})();

(function buildGenreFilter() {
  const genreCount = {};
  nodes.forEach(n => {
    if (n.node_type === "movie" && n.genres) {
      n.genres.split(",").forEach(g => {
        const genre = g.trim();
        if (genre) genreCount[genre] = (genreCount[genre] || 0) + 1;
      });
    }
  });
  const sorted = Object.entries(genreCount).sort((a,b) => b[1]-a[1]);
  const list = document.getElementById("genreList");
  if (!list) return;
  sorted.forEach(([genre, count]) => {
    const lbl = document.createElement("label");
    lbl.style.cssText = "display:flex;align-items:center;gap:6px;cursor:pointer;padding:2px 0;";
    lbl.title = count + " películas";
    lbl.innerHTML = '<input type="checkbox" checked data-genre="' + genre + '">'
      + '<span style="overflow:hidden;text-overflow:ellipsis;">' + genre
      + ' <span style="color:#9ca3af;">(' + count + ')</span></span>';
    list.appendChild(lbl);
    lbl.querySelector("input").addEventListener("change", function () {
      if (this.checked) _filterGenres.delete(genre);
      else _filterGenres.add(genre);
      applyFilters();
    });
  });
})();

function wireListSearch(inputId, listId) {
  const input = document.getElementById(inputId);
  const list = document.getElementById(listId);
  if (!input || !list) return;
  input.addEventListener("input", () => {
    const q = input.value.trim().toLowerCase();
    list.querySelectorAll("label").forEach(lbl => {
      lbl.style.display = (!q || lbl.textContent.toLowerCase().includes(q)) ? "flex" : "none";
    });
  });
}
wireListSearch("countrySearchInput", "countryList");
wireListSearch("genreSearchInput", "genreList");

(function () {
  const btn = document.getElementById("genreToggleAll");
  if (!btn) return;
  let allChecked = true;
  btn.addEventListener("click", function () {
    allChecked = !allChecked;
    btn.textContent = allChecked ? "desmarcar todos" : "marcar todos";
    document.querySelectorAll("#genreList input[type=checkbox]").forEach(cb => {
      cb.checked = allChecked;
      const g = cb.dataset.genre;
      if (allChecked) _filterGenres.delete(g); else _filterGenres.add(g);
    });
    applyFilters();
  });
})();

(function () {
  const btn = document.getElementById("countryToggleAll");
  if (!btn) return;
  let allChecked = true;
  btn.addEventListener("click", function () {
    allChecked = !allChecked;
    btn.textContent = allChecked ? "desmarcar todos" : "marcar todos";
    document.querySelectorAll("#countryList input[type=checkbox]").forEach(cb => {
      cb.checked = allChecked;
      const c = cb.dataset.country;
      if (allChecked) _filterCountries.delete(c); else _filterCountries.add(c);
    });
    applyFilters();
  });
})();

(function () {
  const minS = document.getElementById("yearMin"), maxS = document.getElementById("yearMax");
  const lbl  = document.getElementById("yearRangeLabel");
  if (!minS || !maxS) return;
  function updateYear() {
    let lo = parseInt(minS.value), hi = parseInt(maxS.value);
    if (lo > hi) { const t=lo; lo=hi; hi=t; }
    _filterYearMin = (lo === parseInt(minS.min)) ? null : lo;
    _filterYearMax = (hi === parseInt(maxS.max)) ? null : hi;
    if (lbl) lbl.textContent = lo + "–" + hi;
    applyFilters();
  }
  minS.addEventListener("input", updateYear);
  maxS.addEventListener("input", updateYear);
})();

document.getElementById("exportPosBtn")?.addEventListener("click", exportManualPositions);
document.getElementById("importPosBtn")?.addEventListener("click", () => {
  document.getElementById("importPosInput")?.click();
});
document.getElementById("importPosInput")?.addEventListener("change", function (e) {
  const file = e.target.files && e.target.files[0];
  if (file) importManualPositions(file);
  e.target.value = "";
});

// Un solo boton para las tres cosas (antes eran "Limpiar filtros" y
// "Restablecer posiciones" por separado, y ninguno tocaba el clustering).
function resetEverything() {
  if (!confirm("¿Restablecer filtros, posiciones y clustering?")) return;
  network.unselectAll();
  if (_highlightSet !== null) resetSemantic();
  if (_clusterMode !== "none") setClusterMode("none");
  _filterCountries = new Set(); _filterGenres = new Set();
  _filterYearMin = null; _filterYearMax = null;
  // El grafo arranca con las peliculas OCULTAS (demasiado denso si no);
  // "Restablecer todo" tiene que volver a ese mismo estado, no mostrarlas -
  // aparte de no ser el estado inicial real, mostrar las 849 de golpe
  // dispara la carga de todos sus posters a la vez y se siente lento.
  if (!_hideMovies) edges.add(_virtualDirEdges);
  _hideMovies = true;
  const tmBtn = document.getElementById("toggleMoviesBtn");
  if (tmBtn) { tmBtn.textContent = "Mostrar películas"; tmBtn.classList.add("active"); }
  document.querySelectorAll("#countryList input[type=checkbox], #genreList input[type=checkbox]").forEach(cb => cb.checked = true);
  ["countrySearchInput", "genreSearchInput"].forEach(id => {
    const el = document.getElementById(id);
    if (el) { el.value = ""; el.dispatchEvent(new Event("input")); }
  });
  const minS = document.getElementById("yearMin"), maxS = document.getElementById("yearMax");
  const lbl  = document.getElementById("yearRangeLabel");
  if (minS) minS.value = minS.min;
  if (maxS) maxS.value = maxS.max;
  if (lbl) lbl.textContent = (minS?.min||"") + "–" + (maxS?.max||"");
  if (_yearOrderActive) {
    setYearOrder(false);
    const yoBtn = document.getElementById("toggleYearOrderBtn");
    if (yoBtn) { yoBtn.textContent = "Ordenar por año"; yoBtn.classList.remove("active"); }
  }
  resetManualPositions();
}
document.getElementById("resetAllBtn")?.addEventListener("click", resetEverything);

(function () {
  const btn = document.getElementById("toggleMoviesBtn");
  if (!btn) return;
  btn.addEventListener("click", function () {
    _hideMovies = !_hideMovies;
    btn.textContent = _hideMovies ? "Mostrar películas" : "Ocultar películas";
    btn.classList.toggle("active", _hideMovies);
    if (_hideMovies) {
      edges.add(_virtualDirEdges);
    } else {
      edges.remove(_virtualDirEdges.map(e => e.id));
    }
    applyFilters();
  });
})();

// ======================= ORDEN POR AÑO =======================
// Fija el eje Y de cada director a yearToY(su año) con fixed:{y:true}: la
// física sigue moviendo X libremente, pero Y queda anclado y ordena a los
// directores de mas antiguo (arriba) a mas nuevo (abajo). Año 0/desconocido
// se trata como 1950. Los nodos ya fijados a mano (_manualFixedIds) no se
// tocan, igual que en setClusterMode().
let _yearOrderActive = false;
const YEAR_ORDER_DEFAULT = 1950;

const YEAR_ORDER_X_SPREAD = 4;

function setYearOrder(active) {
  _yearOrderActive = active;
  const updates = [];
  nodes.forEach(n => {
    if (n.node_type !== "director" || _manualFixedIds.has(n.id)) return;
    if (active) {
      const y = (typeof n.year === "number" && !isNaN(n.year) && n.year > 0) ? n.year : YEAR_ORDER_DEFAULT;
      updates.push({ id: n.id, y: yearToY(y), fixed: { x: false, y: true } });
    } else {
      updates.push({ id: n.id, fixed: { x: false, y: false } });
    }
  });
  if (updates.length) nodes.update(updates);
  runBoundedPhysics(() => {
    if (active) {
      // Con Y fijo por año, la separacion X que deja la fisica por defecto
      // (pensada para el layout libre) queda muy apretada entre columnas de
      // la misma decada - se estira al doble una vez asentada.
      const dirIds = nodes.get({ filter: n => n.node_type === "director" && !_manualFixedIds.has(n.id) }).map(n => n.id);
      const pos = network.getPositions(dirIds);
      nodes.update(dirIds.map(id => ({ id, x: pos[id].x * YEAR_ORDER_X_SPREAD })));
    }
    syncMoviePositions();
    updateHeaderStats();
  }, 8000, 400);
}

(function () {
  const btn = document.getElementById("toggleYearOrderBtn");
  if (!btn) return;
  btn.addEventListener("click", function () {
    setYearOrder(!_yearOrderActive);
    btn.textContent = _yearOrderActive ? "Orden libre" : "Ordenar por año";
    btn.classList.toggle("active", _yearOrderActive);
  });
})();
// ======================= HIGHLIGHT =======================
const HIGHLIGHT_EDGE_WIDTH_SCALE = 0.5;

function highlightSemantic(startId) {
  const downstream = getDirectionalNeighborhood(startId, "to");
  const upstream   = getDirectionalNeighborhood(startId, "from");
  _highlightSet = new Set([...downstream.keys(), ...upstream.keys()]);
  // Un nodo puede aparecer en el recorrido y aun así estar excluido por un
  // filtro activo (país/género/año) — sin esto se mostraba igual, junto con
  // sus aristas, aunque el panel de filtros diga que no debería estar.
  // ignoreHideMovies: el highlight existe justamente para revelar películas
  // reales aunque el toggle global "Ocultar películas" esté prendido.
  const hiddenByFilter = computeHiddenByFilter({ ignoreHideMovies: true });

  const nodeUpdates = [];
  nodes.forEach(n => {
    if (isHubNode(n)) return;
    const isActive = _highlightSet.has(n.id) && !hiddenByFilter.has(n.id);
    nodeBaseFontSize[n.id] = isActive ? 26 : 20;
    // El highlight ignora las posiciones movidas a mano (solo aplican en la
    // vista global): todo nodo activo participa de la física normalmente.
    const update = { id: n.id, hidden: !isActive, physics: isActive };
    nodeUpdates.push(update);
  });
  nodes.update(nodeUpdates);
  refreshLabelSizes();

  // Orden natural arriba/abajo: a quienes influyen al nodo elegido (upstream)
  // se les da una posición de partida más arriba, a quienes influye
  // (downstream) más abajo. Dentro de cada franja, los nodos de grado 1 se
  // reparten en x, y cada nodo de grado 2/3 arranca pegado a su ancla de
  // grado 1 (el vecino directo por el que se llegó a él) en vez de repartido
  // por toda la franja — así el racimo de películas que influyen a una
  // película puntual queda cerca de ella, no mezclado con el racimo de otra.
  // No queda fijo (fixed sigue en false arriba) — es solo el punto de
  // partida desde el que la física, que sigue activa, termina de acomodarlos.
  const startNode = nodes.get(startId);
  const baseX = startNode?.x || 0;
  const baseY = startNode?.y || 0;
  const LEVEL_GAP = 260;
  const SIBLING_GAP_1 = 220;
  const SIBLING_GAP_2 = 140;

  function layoutBranch(info, dirSign) {
    const branchUpdates = [];
    const anchorPos = new Map();
    const depth1 = [];
    const byAnchor = new Map();
    info.forEach((meta, id) => {
      if (id === startId) return;
      if (meta.depth === 1) {
        depth1.push(id);
      } else {
        if (!byAnchor.has(meta.anchor)) byAnchor.set(meta.anchor, []);
        byAnchor.get(meta.anchor).push({ id, depth: meta.depth });
      }
    });
    depth1.sort();
    depth1.forEach((id, i) => {
      const x = baseX + (i - (depth1.length - 1) / 2) * SIBLING_GAP_1;
      const y = baseY + dirSign * LEVEL_GAP;
      branchUpdates.push({ id, x, y });
      anchorPos.set(id, { x, y });
    });
    byAnchor.forEach((children, anchorId) => {
      const anchor = anchorPos.get(anchorId);
      if (!anchor) return;
      children.sort((a, b) => a.depth - b.depth || (a.id < b.id ? -1 : 1));
      children.forEach((child, j) => {
        const x = anchor.x + (j - (children.length - 1) / 2) * SIBLING_GAP_2;
        const y = anchor.y + dirSign * (child.depth - 1) * LEVEL_GAP;
        branchUpdates.push({ id: child.id, x, y });
      });
    });
    return branchUpdates;
  }

  const posUpdates = [...layoutBranch(downstream, 1), ...layoutBranch(upstream, -1)];
  if (posUpdates.length) nodes.update(posUpdates);

  // Se muestra toda arista real cuyos dos extremos queden visibles (incluye
  // conexiones paralelas entre el mismo par, ya separadas visualmente por la
  // curvatura) — solo se ocultan las que salen del conjunto resaltado.
  const edgeUpdates = [];
  edges.forEach(e => {
    if (isHubEdge(e)) return;
    // Las aristas virtuales director→director (colapso de películas ocultas)
    // no deben verse en la vista destacada: acá las películas reales ya
    // están visibles, así que la arista colapsada es redundante/engañosa.
    const isVirtual = typeof e.id === "string" && e.id.startsWith("__vdir_");
    const bothActive = !isVirtual
      && _highlightSet.has(e.from) && _highlightSet.has(e.to)
      && !hiddenByFilter.has(e.from) && !hiddenByFilter.has(e.to);
    if (!bothActive) {
      edgeUpdates.push({
        id: e.id,
        color: isVirtual ? "rgba(0,0,0,0)" : "rgba(156,163,175,0.03)",
        dashes: false,
        physics: false,
      });
      return;
    }
    edgeUpdates.push({ id: e.id,
      color: e.base_color || DEFAULT_EDGE_STYLE.color,
      dashes: e.base_dashes || false,
      physics: e.base_physics !== false,
      width: (e.base_width != null ? e.base_width : e.width) * HIGHLIGHT_EDGE_WIDTH_SCALE,
    });
  });
  edges.update(edgeUpdates);

  const activeArr = Array.from(_highlightSet).filter(id => !hiddenByFilter.has(id));
  applyCompactPhysics(() => {
    network.fit({ nodes: activeArr, animation: { duration: 400, easingFunction: "easeInOutQuad" } });
  });
}

function resetSemantic() { _highlightSet = null; applyFilters(); }

network.on("selectNode",   params => { if (params.nodes.length) highlightSemantic(params.nodes[0]); });
network.on("deselectNode", resetSemantic);

// ======================= HOVER: agrandar camino completo =======================
// Con un director seleccionado (_highlightSet activo), las aristas quedan a
// la mitad de su grosor (HIGHLIGHT_EDGE_WIDTH_SCALE, ver highlightSemantic) -
// al pasar el mouse por encima se ve el grosor normal (base_width, sin
// halving) hasta que el mouse se va, momento en que clearHover() tiene que
// saber a qué grosor volver (mitad si segue destacado, completo si no).
const HOVER_NODE_SCALE = 1.6;
const HOVER_EDGE_WIDTH_SCALE = 2.5;
let _hoverActive = null;

function restingEdgeWidth(e) {
  const base = e.base_width != null ? e.base_width : e.width;
  return _highlightSet !== null ? base * HIGHLIGHT_EDGE_WIDTH_SCALE : base;
}

function clearHover() {
  if (!_hoverActive) return;
  const { nodeIds, edgeIds } = _hoverActive;
  nodes.update(nodeIds.map(id => {
    const n = nodes.get(id);
    if (!n) return null;
    return { id, size: n.base_size != null ? n.base_size : n.size };
  }).filter(Boolean));
  edges.update(edgeIds.map(id => {
    const e = edges.get(id);
    if (!e) return null;
    return { id, width: restingEdgeWidth(e) };
  }).filter(Boolean));
  _hoverActive = null;
}

// Recorre el camino completo director -> ... -> director (mismo criterio que
// highlightSemantic, en ambas direcciones) a partir de un director o
// pelicula: no solo sus vecinos directos.
function getFullPathElements(startId) {
  const downstream = getDirectionalNeighborhood(startId, "to");
  const upstream = getDirectionalNeighborhood(startId, "from");
  const nodeIdSet = new Set([...downstream.keys(), ...upstream.keys()]);
  const edgeIdSet = new Set();
  edges.forEach(e => {
    if (isHubEdge(e)) return;
    if (typeof e.id === "string" && e.id.startsWith("__vdir_")) return;
    if (nodeIdSet.has(e.from) && nodeIdSet.has(e.to)) edgeIdSet.add(e.id);
  });
  return { nodeIds: [...nodeIdSet], edgeIds: [...edgeIdSet] };
}

network.on("hoverNode", params => {
  const id = params.node;
  const n = nodes.get(id);
  if (!n || isHubNode(n)) return;
  clearHover();
  const { nodeIds: pathNodeIds, edgeIds: pathEdgeIds } = getFullPathElements(id);
  // Con un director ya destacado (_highlightSet), el hover no puede escaparse
  // a conexiones ajenas a ese camino (ej: destacar Wilder y luego pasar el
  // mouse por Carpenter no tiene que traer a Aldrich solo porque Carpenter y
  // Aldrich estan conectados entre si) - se recorta al set ya visible.
  const scopedNodeIds = _highlightSet !== null
    ? pathNodeIds.filter(nid => _highlightSet.has(nid))
    : pathNodeIds;
  const scopedEdgeIds = _highlightSet !== null
    ? pathEdgeIds.filter(eid => {
        const e = edges.get(eid);
        return e && _highlightSet.has(e.from) && _highlightSet.has(e.to);
      })
    : pathEdgeIds;
  const connectedNodeIds = scopedNodeIds.filter(nid => nid !== id).filter(nid => {
    const nn = nodes.get(nid);
    return nn && !isHubNode(nn) && !nn.hidden;
  });
  const connectedEdgeIds = scopedEdgeIds.filter(eid => {
    const e = edges.get(eid);
    return e && !isHubEdge(e);
  });
  const allNodeIds = [id, ...connectedNodeIds];
  nodes.update(allNodeIds.map(nid => {
    const nn = nodes.get(nid);
    const base = nn.base_size != null ? nn.base_size : nn.size;
    return { id: nid, size: base * HOVER_NODE_SCALE };
  }));
  edges.update(connectedEdgeIds.map(eid => {
    const e = edges.get(eid);
    const base = e.base_width != null ? e.base_width : e.width;
    return { id: eid, width: base * HOVER_EDGE_WIDTH_SCALE };
  }));
  _hoverActive = { nodeIds: allNodeIds, edgeIds: connectedEdgeIds };
});

network.on("blurNode", () => clearHover());

// ======================= AUTOCOMPLETE DE BUSQUEDA =======================
function getSearchMatches(query) {
  if (!query) return [];
  query = query.toLowerCase();
  const results = [];
  nodes.forEach(n => {
    if (isHubNode(n)) return;
    if ((n.label || "").toLowerCase().includes(query)) results.push(n);
  });
  results.sort((a, b) => {
    if (a.node_type !== b.node_type) return a.node_type === "director" ? -1 : 1;
    const aStarts = a.label.toLowerCase().startsWith(query) ? 0 : 1;
    const bStarts = b.label.toLowerCase().startsWith(query) ? 0 : 1;
    if (aStarts !== bStarts) return aStarts - bStarts;
    return a.label.localeCompare(b.label);
  });
  return results.slice(0, 8);
}

let _searchActiveIndex = -1;
let _searchCurrentMatches = [];

function renderSearchResults(matches) {
  _searchCurrentMatches = matches;
  _searchActiveIndex = matches.length ? 0 : -1;
  const box = document.getElementById("searchResults");
  if (!box) return;
  if (!matches.length) { box.style.display = "none"; box.innerHTML = ""; return; }
  box.innerHTML = matches.map((n, i) =>
    '<div class="search-result' + (i === 0 ? ' active' : '') + '" data-idx="' + i + '">'
    + '<span>' + n.label + '</span><span class="sr-type">' + (n.node_type === "director" ? "director" : "película") + '</span>'
    + '</div>'
  ).join("");
  box.style.display = "block";
  box.querySelectorAll(".search-result").forEach((el, i) => {
    // mousedown (no click) + preventDefault: dispara antes que el blur del
    // input, si no el dropdown se cierra antes de registrar el click.
    el.addEventListener("mousedown", (e) => {
      e.preventDefault();
      selectSearchResult(matches[i].id);
    });
  });
}

function selectSearchResult(nodeId) {
  network.selectNodes([nodeId]);
  highlightSemantic(nodeId);
  network.focus(nodeId, { scale: 1.5, animation: { duration: 800, easingFunction: "easeInOutQuad" } });
  renderSearchResults([]);
  const input = document.getElementById("nodeSearch");
  if (input) input.blur();
}

function updateSearchActiveHighlight() {
  const box = document.getElementById("searchResults");
  if (!box) return;
  const items = box.querySelectorAll(".search-result");
  items.forEach((el, i) => el.classList.toggle("active", i === _searchActiveIndex));
  const activeEl = box.querySelector(".search-result.active");
  if (activeEl) activeEl.scrollIntoView({ block: "nearest" });
}

const searchInput = document.getElementById("nodeSearch");
if (searchInput) {
  searchInput.addEventListener("input", () => {
    renderSearchResults(getSearchMatches(searchInput.value));
  });
  searchInput.addEventListener("keydown", (e) => {
    if (e.key === "ArrowDown") {
      if (!_searchCurrentMatches.length) return;
      e.preventDefault();
      _searchActiveIndex = (_searchActiveIndex + 1) % _searchCurrentMatches.length;
      updateSearchActiveHighlight();
    } else if (e.key === "ArrowUp") {
      if (!_searchCurrentMatches.length) return;
      e.preventDefault();
      _searchActiveIndex = (_searchActiveIndex - 1 + _searchCurrentMatches.length) % _searchCurrentMatches.length;
      updateSearchActiveHighlight();
    } else if (e.key === "Enter") {
      if (_searchActiveIndex >= 0 && _searchCurrentMatches[_searchActiveIndex]) {
        selectSearchResult(_searchCurrentMatches[_searchActiveIndex].id);
      }
    } else if (e.key === "Escape") {
      renderSearchResults([]);
      searchInput.blur();
    }
  });
  searchInput.addEventListener("blur", () => {
    setTimeout(() => renderSearchResults([]), 150);
  });
}

// ======================= CLUSTER BUBBLES =======================
const CLUSTER_GROUPS = __CLUSTER_GROUPS_JSON__;
function convexHull(points) {
  if (points.length < 3) return points;
  const pts = points.slice().sort((a,b) => a.x-b.x || a.y-b.y);
  const cross = (o,a,b) => (a.x-o.x)*(b.y-o.y)-(a.y-o.y)*(b.x-o.x);
  const lower = []; for (const p of pts) { while (lower.length>=2 && cross(lower[lower.length-2],lower[lower.length-1],p)<=0) lower.pop(); lower.push(p); }
  const upper = []; for (let i=pts.length-1;i>=0;i--) { const p=pts[i]; while (upper.length>=2 && cross(upper[upper.length-2],upper[upper.length-1],p)<=0) upper.pop(); upper.push(p); }
  upper.pop(); lower.pop(); return lower.concat(upper);
}
function expandHull(hull,padding) {
  if (!hull.length) return hull;
  const cx=hull.reduce((s,p)=>s+p.x,0)/hull.length, cy=hull.reduce((s,p)=>s+p.y,0)/hull.length;
  return hull.map(p=>{const dx=p.x-cx,dy=p.y-cy,dist=Math.sqrt(dx*dx+dy*dy)||1,k=(dist+padding)/dist;return{x:cx+dx*k,y:cy+dy*k};});
}
network.on("beforeDrawing", ctx => {
  if (_clusterMode !== "country") return;
  const pos = network.getPositions();
  Object.values(CLUSTER_GROUPS).forEach(g => {
    const pts=g.nodes.map(id=>{const n=nodes.get(id),p=pos[id];return(!n||n.hidden||!p)?null:{x:p.x,y:p.y};}).filter(Boolean);
    if (!pts.length) return;
    ctx.fillStyle=g.color+"30"; ctx.strokeStyle=g.color+"90"; ctx.lineWidth=2;
    if (pts.length===1){ctx.beginPath();ctx.arc(pts[0].x,pts[0].y,45,0,Math.PI*2);ctx.fill();ctx.stroke();return;}
    if (pts.length===2){ctx.save();ctx.lineWidth=90;ctx.lineCap="round";ctx.strokeStyle=g.color+"30";ctx.beginPath();ctx.moveTo(pts[0].x,pts[0].y);ctx.lineTo(pts[1].x,pts[1].y);ctx.stroke();ctx.restore();return;}
    const hull=expandHull(convexHull(pts),40);
    ctx.beginPath();ctx.moveTo(hull[0].x,hull[0].y);
    for(let i=1;i<hull.length;i++)ctx.lineTo(hull[i].x,hull[i].y);
    ctx.closePath();ctx.fill();ctx.stroke();
  });
});

// 40*500ms=20s tapaba TODA la carga inicial en redraws periodicos (necesario
// porque las imagenes que van llegando async no siempre disparan un repintado
// solas) - con los retratos/posters ahora bastante mas rapidos de bajar
// (menos peso, carga diferida), 8s de margen alcanza de sobra.
(function(){let t=0;const iv=setInterval(()=>{network.redraw();if(++t>=16)clearInterval(iv);},500);})();

// Arrancar con las películas ocultas: el grafo completo (dirs+pelis) es
// demasiado denso para ser legible sin filtrar primero.
edges.add(_virtualDirEdges);
(function () {
  const btn = document.getElementById("toggleMoviesBtn");
  if (btn) { btn.textContent = "Mostrar películas"; btn.classList.add("active"); }
})();
applyFilters();
"""

semantic_js = semantic_js.replace("__CLUSTER_GROUPS_JSON__", cluster_groups_json)
semantic_js = semantic_js.replace("__COUNTRY_COLORS_JSON__", country_colors_json)
semantic_js = semantic_js.replace("__TYPE_STYLES_JSON__", type_styles_json)
semantic_js = semantic_js.replace("__DEFAULT_EDGE_STYLE_JSON__", default_edge_style_json)
semantic_js = semantic_js.replace("__YEAR_MIN_JS__", str(YEAR_MIN))
semantic_js = semantic_js.replace("__YEAR_MAX_JS__", str(YEAR_MAX))

html, n = re.subn(
    r"(new vis.Network\(container, data, options\);)",
    lambda m: m.group(1) + "\n" + semantic_js,
    html,
    count=1
)

assert n == 1, "JS injection failed — pyvis output format may have changed."
print("JS inyectado:", n)

with open("graph_semantic_hover_search.html", "w", encoding="utf-8") as f:
    f.write(html)
print("HTML generado: graph_semantic_hover_search.html")

JS inyectado: 1
HTML generado: graph_semantic_hover_search.html
